In [15]:
import pandas as pd
%run Caminhos.ipynb

tabela_clientes = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_clientes_parquet}')
tabela_ind_estrategico = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_ind_estrategico_parquet}')
tabela_alunos = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_alunos_parquet}')
tabela_especificacao_ind_O_E = pd.read_parquet(f'{pasta_hierarquia}{caminho_especificacao_ind_O_E_parquet}')
tabela_base_gerada = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_base_gerada_parquet}')
tabela_professor = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_professor_parquet}')
tabela_notas = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_notas_parquet}')
tabela_notas_geral = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_notas_geral_parquet}')
tabela_frequencia_ativo = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_frequencia_ativo_parquet}')
tabela_frequencia_ITA = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_frequencia_ITA_parquet}')
tabela_Tempo_Integral = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_Tempo_Integral_parquet}')

Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/


85 - % de estudantes da rede estadual com frequência regular em 75%

In [211]:
tab_qtd_freq = tabela_frequencia_ativo

## Soma as frequencias dos estudantes inscritos nas turmas ITA/IME
freq_reg = tab_qtd_freq.merge(tabela_clientes[["idMatricula", "Id Municipio", "Matricula_2024", "Matricula_2025", "Matricula_2026"]], on='idMatricula', how='left')
freq_reg = freq_reg.drop_duplicates(subset=["idMatricula", "Id Municipio"])
freq_reg_aprov = freq_reg[freq_reg['percPresenca'] >= 75]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
freq_apr_24 = freq_reg_aprov[freq_reg_aprov['Matricula_2024'] == 1]
freq_apr_25 = freq_reg_aprov[freq_reg_aprov['Matricula_2025'] == 1]

soma_freq_apr_24 = freq_apr_24.groupby('Id Municipio')['idMatricula'].count().reset_index()
soma_freq_apr_25 = freq_apr_25.groupby('Id Municipio')['idMatricula'].count().reset_index()

# Adicionar coluna do ano
soma_freq_apr_24['Ano'] = 2024
soma_freq_apr_25['Ano'] = 2025

# Concatenar os DataFrames
soma_freq_apr = pd.concat([soma_freq_apr_24, soma_freq_apr_25])
#display(soma_freq_apr)
'----------------------------------------------------------------------------------------------------------------------------------------'
freq_reg_reprov = freq_reg[freq_reg['percPresenca'] < 75]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
freq_rep_24 = freq_reg_reprov[freq_reg_reprov['Matricula_2024'] == 1]
freq_rep_25 = freq_reg_reprov[freq_reg_reprov['Matricula_2025'] == 1]

soma_freq_rep_24 = freq_rep_24.groupby('Id Municipio')['idMatricula'].count().reset_index()
soma_freq_rep_25 = freq_rep_25.groupby('Id Municipio')['idMatricula'].count().reset_index()

# Adicionar coluna do ano
soma_freq_rep_24['Ano'] = 2024
soma_freq_rep_25['Ano'] = 2025

# Concatenar os DataFrames
soma_freq_rep = pd.concat([soma_freq_rep_24, soma_freq_rep_25])
#display(soma_freq_rep)
'----------------------------------------------------------------------------------------------------------------------------------------'
est_freq = pd.concat([soma_freq_apr, soma_freq_rep])
#display(est_freq)
'----------------------------------------------------------------------------------------------------------------------------------------'
# Matriculas totais
sum_est_freq = est_freq.groupby(['Id Municipio', 'Ano'])['idMatricula'].sum().reset_index()
sum_est_freq = sum_est_freq.rename(columns={'idMatricula': 'Qtde_Total'})
#display(sum_est_freq)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
freq_aprovativa = soma_freq_apr.merge(sum_est_freq, on=['Id Municipio', 'Ano'])
freq_aprovativa['% de estudantes da rede estadual com frequência regular em 75%'] = freq_aprovativa['idMatricula'] / freq_aprovativa['Qtde_Total']
freq_aprovativa = freq_aprovativa[["Id Municipio", "Ano", "% de estudantes da rede estadual com frequência regular em 75%"]]
display(freq_aprovativa)

,Id Municipio,Ano,% de estudantes da rede estadual com frequência regular em 75%
0,2200053,2024,0.957346
1,2200103,2024,0.961415
2,2200202,2024,0.903333
3,2200251,2024,0.928870
4,2200277,2024,0.963158
...,...,...,...
219,2211357,2024,0.960699
220,2211407,2024,0.948980
221,2211506,2024,0.954667
222,2211605,2024,0.994580


385 - Indicador - % frequência dos estudantes inscritos nas turmas ITA/IME

In [212]:
tab_freq = tabela_frequencia_ITA

## Qtd aulas com presença dos estudantes inscritos nas turmas ITA/IME
tab_freq = tab_freq.merge(tabela_clientes[["idMatricula", "Id Municipio", "Matricula_2024", "Matricula_2025", "Matricula_2026"]], on='idMatricula', how='left')

# Filtrar o DataFrame onde tipoFrequencia = Presença
tab_freq_p = tab_freq[(tab_freq['tipoFrequencia'] == "Presença")]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
freq_24 = tab_freq_p[tab_freq_p['Matricula_2024'] == 1]
freq_25 = tab_freq_p[tab_freq_p['Matricula_2025'] == 1]

soma_freq_24 = freq_24.groupby(['Id Municipio', 'Ano'])['idAula'].count().reset_index()
soma_freq_25 = freq_25.groupby(['Id Municipio', 'Ano'])['idAula'].count().reset_index()

# Concatenar os DataFrames
soma_freq = pd.concat([soma_freq_24, soma_freq_25])
soma_freq = soma_freq.rename(columns={'idAula': 'Qtde aulas com presença'})
soma_freq['Ano'] = soma_freq['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras
freq_24_total = tab_freq[tab_freq['Matricula_2024'] == 1]
freq_25_total = tab_freq[tab_freq['Matricula_2025'] == 1]

# Contar o número de idMatriculas distintas por Id Estado
estudantes_freq_24  = freq_24_total.groupby(['Id Municipio', 'Ano'])['idAula'].count().reset_index()
estudantes_freq_25  = freq_25_total.groupby(['Id Municipio', 'Ano'])['idAula'].count().reset_index()

# Concatenar os DataFrames
estudantes_freq = pd.concat([estudantes_freq_24, estudantes_freq_25])
estudantes_freq = estudantes_freq.rename(columns={'idAula': 'Qtde aulas'})
estudantes_freq['Ano'] = estudantes_freq['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_freq = soma_freq.merge(estudantes_freq, on=['Id Municipio', 'Ano'])
base_nota_freq['% frequência dos estudantes inscritos nas turmas ITA/IME'] = base_nota_freq['Qtde aulas com presença'] / base_nota_freq['Qtde aulas']
base_nota_freq = base_nota_freq[["Id Municipio", "Ano", "% frequência dos estudantes inscritos nas turmas ITA/IME"]]
display(base_nota_freq)

,Id Municipio,Ano,% frequência dos estudantes inscritos nas turmas ITA/IME
0,2211001,2024,0.999833


381 - Indicador - Média nas avaliações nas demais disciplinas

In [213]:
tab_nota_ITA = tabela_notas_geral[["Id Municipio", "idMatricula", "Ano", "idTurma", "nomeDisciplina", "nomeTipoNota", "valorNota", "idDisciplina", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular

# Filtrar o DataFrame onde turma = ITA/IME
tab_nota_ITA = tab_nota_ITA[(tab_nota_ITA['idTurma'].isin([287816, 287818]))]
# Filtrar o DataFrame onde nomeDisciplina
tab_nota_ITA = tab_nota_ITA[(~tab_nota_ITA['idDisciplina'].isin([2551, 2556, 2557, 2558, 2559, 2560, 2561, 2562, 2563, 2564, 2565, 2566, 2567, 594, 2483, 2554, 595]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_ITA_24 = tab_nota_ITA[tab_nota_ITA['Matricula_2024'] == 1]
nota_ITA_25 = tab_nota_ITA[tab_nota_ITA['Matricula_2025'] == 1]

soma_nota_ITA_24 = nota_ITA_24.groupby(['Id Municipio', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_ITA_25 = nota_ITA_25.groupby(['Id Municipio', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_ITA = pd.concat([soma_nota_ITA_24, soma_nota_ITA_25])
soma_nota_ITA['Ano'] = soma_nota_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras

# Contar o número de idMatriculas distintas por Id Estado
estudantes_ITA_24  = nota_ITA_24.groupby(['Id Municipio', 'Ano'])['idMatricula'].count().reset_index()
estudantes_ITA_25  = nota_ITA_25.groupby(['Id Municipio', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_ITA = pd.concat([estudantes_ITA_24, estudantes_ITA_25])
estudantes_ITA['Ano'] = estudantes_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_ITA = soma_nota_ITA.merge(estudantes_ITA, on=['Id Municipio', 'Ano'])
base_nota_media_ITA['Média nas avaliações nas demais disciplinas'] = base_nota_media_ITA['valorNota'] / base_nota_media_ITA['idMatricula']

base_nota_media_ITA_IME = base_nota_media_ITA.drop(['valorNota', 'idMatricula'], axis=1)
#base_nota_media_ITA_IME = base_nota_media_ITA_IME[(base_nota_media_ITA_IME["Id Municipio"] == 3)]
display(base_nota_media_ITA_IME)

,Id Municipio,Ano,Média nas avaliações nas demais disciplinas
0,2211001,2024,7.554592


382 - Indicador - Média nas avaliações das disciplinas especificas ITA/IME

In [214]:
tab_nota_esp_ITA = tabela_notas_geral[["Id Municipio", "idMatricula", "idTurma", "Ano", "nomeDisciplina", "nomeTipoNota", "valorNota", "idDisciplina", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular
# Filtrar o DataFrame onde turma = ITA/IME
tab_nota_esp_ITA = tab_nota_esp_ITA[(tab_nota_esp_ITA['idTurma'].isin([287816, 287818]))]
# Filtrar o DataFrame onde nomeDisciplina
tab_nota_esp_ITA = tab_nota_esp_ITA[(tab_nota_esp_ITA['idDisciplina'].isin([2551, 2556, 2557, 2558, 2559, 2560, 2561, 2562, 2563, 2564, 2565, 2566, 2567, 594, 2483, 2554, 595]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_esp_ITA_24 = tab_nota_esp_ITA[tab_nota_esp_ITA['Matricula_2024'] == 1]
nota_esp_ITA_25 = tab_nota_esp_ITA[tab_nota_esp_ITA['Matricula_2025'] == 1]

soma_nota_esp_ITA_24 = nota_esp_ITA_24.groupby(['Id Municipio', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_esp_ITA_25 = nota_esp_ITA_25.groupby(['Id Municipio', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_esp_ITA = pd.concat([soma_nota_esp_ITA_24, soma_nota_esp_ITA_25])
soma_nota_esp_ITA['Ano'] = soma_nota_esp_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras

# Contar o número de idMatriculas distintas por Id Estado
estudantes_esp_ITA_24  = nota_esp_ITA_24.groupby(['Id Municipio', 'Ano'])['idMatricula'].count().reset_index()
estudantes_esp_ITA_25  = nota_esp_ITA_25.groupby(['Id Municipio', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_esp_ITA = pd.concat([estudantes_esp_ITA_24, estudantes_esp_ITA_25])
estudantes_esp_ITA['Ano'] = estudantes_esp_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_esp_ITA = soma_nota_esp_ITA.merge(estudantes_esp_ITA, on=['Id Municipio', 'Ano'])
base_nota_media_esp_ITA['Média nas avaliações das disciplinas especificas ITA/IME'] = base_nota_media_esp_ITA['valorNota'] / base_nota_media_esp_ITA['idMatricula']

base_nota_media_esp_ITA_IME = base_nota_media_esp_ITA.drop(['valorNota', 'idMatricula'], axis=1)
#base_nota_media_esp_ITA_IME = base_nota_media_esp_ITA_IME[(base_nota_media_esp_ITA_IME["Id Municipio"] == 3)]
display(base_nota_media_esp_ITA_IME)

,Id Municipio,Ano,Média nas avaliações das disciplinas especificas ITA/IME
0,2211001,2024,7.319304


402 - Indicador - Média dos alunos da rede nas disciplinas de línguas estrangeiras

In [215]:
tab_nota_LEst = tabela_notas_geral[["Id Municipio", "idMatricula", "Ano", "nomeDisciplina", "nomeTipoNota", "valorNota", "idDisciplina", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular Linguas Estrangeiras

# Filtrar o DataFrame onde nomeDisciplina = LÍNGUAS ESTRANGEIRAS
nota_m_LEst = tab_nota_LEst[(tab_nota_LEst['idDisciplina'].isin([2400, 2233, 2232, 11, 10, 9]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_LEst_24 = nota_m_LEst[nota_m_LEst['Matricula_2024'] == 1]
nota_LEst_25 = nota_m_LEst[nota_m_LEst['Matricula_2025'] == 1]

soma_nota_LEst_24 = nota_LEst_24.groupby(['Id Municipio', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_LEst_25 = nota_LEst_25.groupby(['Id Municipio', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_LEst = pd.concat([soma_nota_LEst_24, soma_nota_LEst_25])
soma_nota_LEst['Ano'] = soma_nota_LEst['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras

# Contar o número de idMatriculas distintas por Id Estado
estudantes_LEst_24  = nota_LEst_24.groupby(['Id Municipio', 'Ano'])['idMatricula'].count().reset_index()
estudantes_LEst_25  = nota_LEst_25.groupby(['Id Municipio', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_LEst = pd.concat([estudantes_LEst_24, estudantes_LEst_25])
soma_nota_LEst['Ano'] = soma_nota_LEst['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_LEst = soma_nota_LEst.merge(estudantes_LEst, on=['Id Municipio', 'Ano'])
base_nota_media_LEst['Média dos alunos da rede nas disciplinas de línguas estrangeiras'] = base_nota_media_LEst['valorNota'] / base_nota_media_LEst['idMatricula']

base_nota_media_LP = base_nota_media_LEst.drop(['valorNota', 'idMatricula'], axis=1)
#base_nota_media_LP = base_nota_media_LP[(base_nota_media_LP["Id Municipio"] == 3)]
display(base_nota_media_LP)

,Id Municipio,Ano,Média dos alunos da rede nas disciplinas de línguas estrangeiras
0,2200053,2024,7.978935
1,2200103,2024,7.209310
2,2200202,2024,7.039017
3,2200251,2024,8.250000
4,2200277,2024,7.377064
...,...,...,...
216,2211357,2024,6.181788
217,2211407,2024,7.312811
218,2211506,2024,5.969853
219,2211605,2024,7.524171


163 - Indicador - % de aprovação dos alunos do Ensino Médio - ok - A Validar

In [216]:
tab_nota_EM_Aprov_geral = tabela_notas_geral[["Id Municipio", "idMatricula", "Ano", "nomeTipoNota", "valorNota", "Matricula_2024", "Matricula_2025"]]
tab_client_EM_Aprov = tabela_clientes[["idMatricula", "Id Municipio", "Ano", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tab_nota_EM_Aprov_geral = tab_nota_EM_Aprov_geral.merge(tab_client_EM_Aprov, on=['idMatricula', 'Id Municipio', 'Ano'], how='inner')

#Filtro 'Ensino Médio' e nomeDisciplina = LÍNGUA PORTUGUESA
filt_EM = tab_nota_EM_Aprov_geral[
    (tab_nota_EM_Aprov_geral['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tab_nota_EM_Aprov_geral['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tab_nota_EM_Aprov_geral['grupo'].str.contains('Ensino', regex=False))
    ]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filt_EM_24 = filt_EM[filt_EM['Matricula_2024'] == 1]
filt_EM_25 = filt_EM[filt_EM['Matricula_2025'] == 1]

# Agrupando por "Id Escola" e "idMatricula" e calculando a média de "valorNota"
agrupar_media_24 = filt_EM_24.groupby(["Id Municipio", "Ano", "idMatricula", "nomeTipoNota"])["valorNota"].mean().reset_index()
agrupar_media_25 = filt_EM_25.groupby(["Id Municipio", "Ano", "idMatricula", "nomeTipoNota"])["valorNota"].mean().reset_index()

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [agrupar_media_25, agrupar_media_24]
mat_EM_geral = pd.concat([contar_distintos(df) for df in dataframes])

mat_EM_geral = mat_EM_geral.rename(columns={'idMatricula': 'Qtde_Matriculas'})
"-----------------------------------------------------------------------------------------------------------------"
# Filtrar o DataFrame onde agregada = Ensino Médio
filt_EM_aprov_geral_24 = agrupar_media_24[((agrupar_media_24['valorNota'] >= 6))]
filt_EM_aprov_geral_25 = agrupar_media_25[((agrupar_media_25['valorNota'] >= 6))]

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_EM_aprov_geral_25, filt_EM_aprov_geral_24]
aprovados_EM_geral = pd.concat([contar_distintos(df) for df in dataframes])

aprovados_EM_geral = aprovados_EM_geral.rename(columns={'idMatricula': 'Qtde_Matriculas_Aprovadas'})
"-----------------------------------------------------------------------------------------------------------------"
# Mesclar os dois DataFrames pelo "Id Escola"
EM_geral = mat_EM_geral.merge(aprovados_EM_geral, on=['Id Municipio', 'Ano'])
EM_geral['Ano'] = EM_geral['Ano'].astype(int)
"-----------------------------------------------------------------------------------------------------------------"
# % de aprovação dos alunos do Ensino Médio
EM_geral['% de aprovação dos alunos do Ensino Médio'] = EM_geral['Qtde_Matriculas_Aprovadas'] / EM_geral['Qtde_Matriculas']
#EM_geral = mat_EM_geral[mat_EM_geral["Id Escola"] == 3]
EM_geral = EM_geral[['Id Municipio', 'Ano', '% de aprovação dos alunos do Ensino Médio']]
display(EM_geral)

,Id Municipio,Ano,% de aprovação dos alunos do Ensino Médio
0,2200053,2024,0.957346
1,2200103,2024,0.939394
2,2200202,2024,0.930830
3,2200251,2024,0.960526
4,2200277,2024,0.912162
...,...,...,...
219,2211357,2024,0.863354
220,2211407,2024,0.879195
221,2211506,2024,0.905660
222,2211605,2024,0.900000


1193 - Indicador - % de estudantes do E.M. com média >= 6 no componente curricular Matemática - ok - Validado

In [217]:
tab_nota_M_Aprovados = tabela_notas[["Id Municipio", "idMatricula", "Ano", "mediaNota", "nomeDisciplina", "Matricula_2024", "Matricula_2025"]]
tab_client_M_Aprovados = tabela_clientes[["idMatricula", "Id Municipio", "Ano", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tab_nota_M_Aprovados = tab_nota_M_Aprovados.merge(tab_client_M_Aprovados, on=['idMatricula', 'Id Municipio', 'Ano'], how='inner')

#Filtro 'Ensino Médio' e nomeDisciplina = MATEMÁTICA
filt_M = tab_nota_M_Aprovados[
    (tab_nota_M_Aprovados['nomeDisciplina'] == "MATEMÁTICA") & 
    (tab_nota_M_Aprovados['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tab_nota_M_Aprovados['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tab_nota_M_Aprovados['grupo'].str.contains('Ensino', regex=False))
    ]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filt_M_24 = filt_M[filt_M['Matricula_2024'] == 1]
filt_M_25 = filt_M[filt_M['Matricula_2025'] == 1]

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_M_25, filt_M_24]
soma_nota_M = pd.concat([contar_distintos(df) for df in dataframes])
"-----------------------------------------------------------------------------------------------------------------"
# Filtrar o DataFrame onde agregada = Ensino Médio
filt_M_aprovados_24 = filt_M_24[((filt_M_24['mediaNota'] >= 6))]
filt_M_aprovados_25 = filt_M_25[((filt_M_25['mediaNota'] >= 6))]

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_M_aprovados_25, filt_M_aprovados_24]
aprovados_M = pd.concat([contar_distintos(df) for df in dataframes])

# Renomear a coluna para identificação clara
aprovados_M = aprovados_M.rename(columns={'idMatricula': 'Qtde_Matriculas_Aprovadas'})
"-----------------------------------------------------------------------------------------------------------------"
# Mesclar os dois DataFrames pelo "Id Municipio"
mat_M = soma_nota_M.merge(aprovados_M, on=['Id Municipio', 'Ano'])
mat_M['Ano'] = mat_M['Ano'].astype(int)

# Substituir valores NaN por 0 na nova coluna (caso uma escola não tenha nenhuma matrícula com nota >= 6)
mat_M['Qtde_Matriculas_Aprovadas'] = mat_M['Qtde_Matriculas_Aprovadas'].fillna(0)
"-----------------------------------------------------------------------------------------------------------------"
# % de estudantes do E.M. com média >= 6 no componente curricular Matemática
mat_M['% de estudantes do E.M. com média >= 6 no componente curricular Matemática'] = mat_M['Qtde_Matriculas_Aprovadas'] / mat_M['idMatricula']
mat_M = mat_M[["Id Municipio", "Ano", "% de estudantes do E.M. com média >= 6 no componente curricular Matemática"]]
display(mat_M)

,Id Municipio,Ano,% de estudantes do E.M. com média >= 6 no componente curricular Matemática
0,2200053,2024,0.902439
1,2200103,2024,0.943503
2,2200202,2024,0.914807
3,2200251,2024,0.947368
4,2200277,2024,0.902098
...,...,...,...
218,2211357,2024,0.626582
219,2211407,2024,0.727891
220,2211506,2024,0.686275
221,2211605,2024,0.750000


1194 - Indicador - % de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa - ok - Validado

In [218]:
tab_nota_LP_Aprovados = tabela_notas[["Id Municipio", "idMatricula", "Ano", "mediaNota", "nomeDisciplina", "Matricula_2024", "Matricula_2025"]]
tab_client_LP_Aprovados = tabela_clientes[["idMatricula", "Id Municipio", "Ano", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tab_nota_LP_Aprovados = tab_nota_LP_Aprovados.merge(tab_client_LP_Aprovados, on=['idMatricula', 'Id Municipio', 'Ano'], how='inner')

#Filtro 'Ensino Médio' e nomeDisciplina = LÍNGUA PORTUGUESA
filt_LP = tab_nota_LP_Aprovados[
    (tab_nota_LP_Aprovados['nomeDisciplina'] == "LÍNGUA PORTUGUESA") & 
    (tab_nota_LP_Aprovados['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tab_nota_LP_Aprovados['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tab_nota_LP_Aprovados['grupo'].str.contains('Ensino', regex=False))
    ]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filt_LP_24 = filt_LP[filt_LP['Matricula_2024'] == 1]
filt_LP_25 = filt_LP[filt_LP['Matricula_2025'] == 1]

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_LP_25, filt_LP_24]
soma_nota_LP = pd.concat([contar_distintos(df) for df in dataframes])
"-----------------------------------------------------------------------------------------------------------------"
# Filtrar o DataFrame onde agregada = Ensino Médio
filt_LP_aprovados_24 = filt_LP_24[((filt_LP_24['mediaNota'] >= 6))]
filt_LP_aprovados_25 = filt_LP_25[((filt_LP_25['mediaNota'] >= 6))]

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_LP_aprovados_25, filt_LP_aprovados_24]
aprovados_nota_LP = pd.concat([contar_distintos(df) for df in dataframes])

aprovados_nota_LP = aprovados_nota_LP.rename(columns={'idMatricula': 'Qtde_Matriculas_Aprovadas'})
"-----------------------------------------------------------------------------------------------------------------"
# Mesclar os dois DataFrames pelo "Id Municipio"
nota_LP = soma_nota_LP.merge(aprovados_nota_LP, on=['Id Municipio', 'Ano'])
nota_LP['Ano'] = nota_LP['Ano'].astype(int)
"-----------------------------------------------------------------------------------------------------------------"
# % de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa
nota_LP['% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa'] = nota_LP['Qtde_Matriculas_Aprovadas'] / nota_LP['idMatricula']
#mat_LP = mat_LP[['Id Estado', '% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa']]
nota_LP = nota_LP[["Id Municipio", "Ano", "% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa"]]
display(nota_LP)

,Id Municipio,Ano,% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa
0,2200053,2024,0.988372
1,2200103,2024,0.943878
2,2200202,2024,0.932609
3,2200251,2024,0.933775
4,2200277,2024,0.780142
...,...,...,...
216,2211357,2024,0.690323
217,2211407,2024,0.862069
218,2211506,2024,0.644231
219,2211605,2024,0.909091


2237 - Indicador - Nota média dos estudantes da Rede no componente curricular Língua Portuguesa

In [219]:
tab_nota_LP = tabela_notas_geral[["Id Municipio", "idMatricula", "Ano", "idDisciplina", "agregada", "nomeTipoNota", "valorNota", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular LP

# Filtrar o DataFrame onde nomeDisciplina = LÍNGUA PORTUGUESA
nota_m_LP = tab_nota_LP[(tab_nota_LP['idDisciplina'] == 1)]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_LP_24 = nota_m_LP[nota_m_LP['Matricula_2024'] == 1]
nota_LP_25 = nota_m_LP[nota_m_LP['Matricula_2025'] == 1]

soma_nota_LP_24 = nota_LP_24.groupby(['Id Municipio', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_LP_25 = nota_LP_25.groupby(['Id Municipio', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_LP = pd.concat([soma_nota_LP_24, soma_nota_LP_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente LP

# Contar o número de idMatriculas distintas por Id Municipio
estudantes_LP_24  = nota_LP_24.groupby(['Id Municipio', 'Ano'])['idMatricula'].count().reset_index()
estudantes_LP_25  = nota_LP_25.groupby(['Id Municipio', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_LP = pd.concat([estudantes_LP_24, estudantes_LP_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
nota_media_LP = soma_nota_LP.merge(estudantes_LP, on=['Id Municipio', 'Ano'])
nota_media_LP['Ano'] = nota_media_LP['Ano'].astype(int)
nota_media_LP['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'] = nota_media_LP['valorNota'] / nota_media_LP['idMatricula']

nota_media_LP = nota_media_LP.drop(['valorNota', 'idMatricula'], axis=1)
#nota_media_LP = nota_media_LP[(nota_media_LP["Id Municipio"] == 3)]
display(nota_media_LP)

,Id Municipio,Ano,Nota média dos estudantes da Rede no componente curricular Língua Portuguesa
0,2200053,2024,6.875182
1,2200103,2024,7.062472
2,2200202,2024,7.459429
3,2200251,2024,7.114249
4,2200277,2024,5.933948
...,...,...,...
216,2211357,2024,6.006667
217,2211407,2024,6.564014
218,2211506,2024,5.958738
219,2211605,2024,6.329545


2238 - Indicador - Nota média dos estudantes da Rede no componente curricular Matemática - OK

In [220]:
tab_nota_M = tabela_notas_geral[["Id Municipio", "idMatricula", "Ano", "idDisciplina", "agregada", "nomeTipoNota", "valorNota", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular M

# Filtrar o DataFrame onde nomeDisciplina = LÍNGUA PORTUGUESA
nota_M = tab_nota_M[(tab_nota_M['idDisciplina'] == 5)]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_M_24 = nota_M[nota_M['Matricula_2024'] == 1]
nota_M_25 = nota_M[nota_M['Matricula_2025'] == 1]

soma_nota_M_24 = nota_M_24.groupby(['Id Municipio', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_M_25 = nota_M_25.groupby(['Id Municipio', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_M = pd.concat([soma_nota_M_24, soma_nota_M_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente M

# Contar o número de idMatriculas distintas por Id Municipio
estudantes_M_24 = nota_M_24.groupby(['Id Municipio', 'Ano'])['idMatricula'].count().reset_index()
estudantes_M_25 = nota_M_25.groupby(['Id Municipio', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_M = pd.concat([estudantes_M_24, estudantes_M_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_M = soma_nota_M.merge(estudantes_M, on=['Id Municipio', 'Ano'])
base_nota_media_M['Ano'] = base_nota_media_M['Ano'].astype(int)

base_nota_media_M['Nota média dos estudantes da Rede no componente curricular Matemática'] = base_nota_media_M['valorNota'] / base_nota_media_M['idMatricula']
base_nota_media_M = base_nota_media_M.drop(['valorNota', 'idMatricula'], axis=1)
display(base_nota_media_M)

,Id Municipio,Ano,Nota média dos estudantes da Rede no componente curricular Matemática
0,2200053,2024,7.115642
1,2200103,2024,6.672155
2,2200202,2024,6.505238
3,2200251,2024,7.118546
4,2200277,2024,6.991459
...,...,...,...
218,2211357,2024,5.901974
219,2211407,2024,5.969416
220,2211506,2024,5.679397
221,2211605,2024,6.357955


2227 - Indicador - Nº de docentes autodeclarados pretos/pardos na rede estadual

2225 - Indicador - Nº de docentes autodeclarados indígenas na rede estadual

2215 - Indicador - N° de estudantes autodeclarados pretos matriculados na rede - ok

In [221]:
tabela_alunos_est = tabela_alunos[["idAluno", "codigoEscolarAluno", "RA", "nome", "sexo", "raca"]]

# Filtrar alunos com raca 'Preta'
tabela_alunos_preta = tabela_alunos_est[tabela_alunos_est['raca'] == 'Preta']

# Mesclar os DataFrames para adicionar as colunas com o cálculo
Matricula_Alunos = tabela_alunos_preta.merge(tabela_clientes[["idAluno", "idMatricula", "Id Municipio", "Matricula_2026", "Matricula_2025", "Matricula_2024", "Ano"]], on=['idAluno'], how='inner')

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filter_df_24 = Matricula_Alunos[Matricula_Alunos['Matricula_2024'] == 1]
filter_df_25 = Matricula_Alunos[Matricula_Alunos['Matricula_2025'] == 1]

# Contar os valores distintos na coluna idMatricula por municipio (idMunicipio)
distinct_count_Preto_24_mun = filter_df_24.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()
distinct_count_Preto_25_mun = filter_df_25.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames
matricula_aut_pretos = pd.concat([distinct_count_Preto_24_mun, distinct_count_Preto_25_mun])
matricula_aut_pretos.rename(columns={'idMatricula': 'N° de estudantes autodeclarados pretos matriculados na rede'}, inplace=True)
matricula_aut_pretos['Ano'] = matricula_aut_pretos['Ano'].astype(int)
display(matricula_aut_pretos)

,Id Municipio,Ano,N° de estudantes autodeclarados pretos matriculados na rede
0,2200053,2024,17
1,2200103,2024,8
2,2200202,2024,31
3,2200251,2024,4
4,2200301,2024,12
...,...,...,...
199,2211308,2024,53
200,2211357,2024,10
201,2211407,2024,13
202,2211605,2024,8


2196 - Indicador - N° de estudantes autodeclarados indígenas matriculados na rede - ok - A Validar

In [222]:
# Filtrar alunos com raca 'Preta'
tabela_alunos_Ind = tabela_alunos_est[tabela_alunos_est['raca'] == 'Indígena']

num_linhas = tabela_alunos_est.shape[0]

# Mesclar os DataFrames para adicionar as colunas com o cálculo
Matricula_Alunos_Ind = tabela_alunos_Ind.merge(tabela_clientes[["idAluno", "idMatricula", "Id Municipio", "Matricula_2026", "Matricula_2025", "Matricula_2024", "Ano"]], on=['idAluno'], how='inner')

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filter_Ind_df_24 = Matricula_Alunos_Ind[Matricula_Alunos_Ind['Matricula_2024'] == 1]
filter_Ind_df_25 = Matricula_Alunos_Ind[Matricula_Alunos_Ind['Matricula_2025'] == 1]

# Contar os valores distintos na coluna idMatricula por escola (Id Escola)
idMatricula_Ind_24_escola = filter_Ind_df_24.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()
idMatricula_Ind_25_escola = filter_Ind_df_25.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames
matricula_aut_indigena = pd.concat([idMatricula_Ind_24_escola, idMatricula_Ind_25_escola])
matricula_aut_indigena.rename(columns={'idMatricula': 'N° de estudantes autodeclarados indígenas matriculados na rede'}, inplace=True)
matricula_aut_indigena['Ano'] = matricula_aut_indigena['Ano'].astype(int)
display(matricula_aut_indigena)

,Id Municipio,Ano,N° de estudantes autodeclarados indígenas matriculados na rede
0,2200202,2024,1
1,2200509,2024,2
2,2201150,2024,3
3,2201200,2024,1
4,2201903,2024,1
5,2202026,2024,1
6,2203230,2024,2
7,2203305,2024,1
8,2204402,2024,2
9,2205508,2024,1


153 - Indicador - % de municípios com escolas da rede estadual em tempo integral

In [223]:
# Qtde de municipio com escolas da rede estadual
tabela_clientes_matricula_est = tabela_clientes[["idMatricula", "Id Estado", "Id Municipio", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
mun_24 = tabela_clientes_matricula_est[(tabela_clientes_matricula_est['Matricula_2024'] == 1)]
mun_24 = mun_24[["Id Estado", "Id Municipio", "Ano"]]
mun_25 = tabela_clientes_matricula_est[(tabela_clientes_matricula_est['Matricula_2025'] == 1)]
mun_25 = mun_25[["Id Estado", "Id Municipio", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (Id Estado)
mun_24['Qtde de municipio com escolas da rede estadual'] = 1
mun_25['Qtde de municipio com escolas da rede estadual'] = 1

# Concatenar os DataFrames
combined_mun = pd.concat([mun_24, mun_25])

# Remover duplicatas com base nas colunas 'Id Municipio' e 'Ano'
combined_mun = combined_mun.drop('Id Estado', axis=1)
combined_mun = combined_mun.drop_duplicates(keep='first')

combined_mun['Ano'] = combined_mun['Ano'].astype(int)
display(combined_mun)

,Id Municipio,Ano,Qtde de municipio com escolas da rede estadual
3,2202091,2024,1
29381,2209302,2024,1
58691,2211001,2024,1
59413,2202737,2024,1
146018,2209906,2024,1
...,...,...,...
477788,2209450,2024,1
478100,2201988,2024,1
478321,2200053,2024,1
497844,2205151,2024,1


In [224]:
# Quantidade de municipio com escolas da rede estadual em tempo integral
tabela_mun_integral = tabela_Tempo_Integral[["Id Estado", "Id Escola", "Id Municipio", "Ano", "STATUS_INTEGRAL"]]

filt_status_mun = tabela_mun_integral[tabela_mun_integral['STATUS_INTEGRAL'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_24_mun = filt_status_mun[filt_status_mun['Ano'] == 2024]
filt_24_mun = filt_24_mun[["Id Estado", "Id Municipio", "Ano"]]
filt_25_mun = filt_status_mun[filt_status_mun['Ano'] == 2025]
filt_25_mun = filt_25_mun[["Id Estado", "Id Municipio", "Ano"]]

filt_24_mun['Qtde de Municipio com escolas de tempo integral'] = 1
filt_25_mun['Qtde de Municipio com escolas de tempo integral'] = 1

# Concatenar os Dataframes
comb_int_mun = pd.concat([filt_24_mun, filt_25_mun])

# Remover duplicatas com base nas colunas 'Id Municipio' e 'Ano'
comb_int_mun = comb_int_mun.drop('Id Estado', axis=1)
comb_int_mun = comb_int_mun.drop_duplicates(keep='first')
comb_int_mun['Ano'] = comb_int_mun['Ano'].astype(int)
"----------------------------------------------------------------------------------------------------------------"
# Mesclagem para DF final
resultado_mun = combined_mun.merge(comb_int_mun, on=['Id Municipio', 'Ano'], how='inner')
"----------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
resultado_mun['% de municípios com escolas da rede estadual em tempo integral'] = resultado_mun['Qtde de Municipio com escolas de tempo integral'] / resultado_mun['Qtde de municipio com escolas da rede estadual']
resultado_mun = resultado_mun[["Id Municipio", "% de municípios com escolas da rede estadual em tempo integral", "Ano"]]
display(resultado_mun)

,Id Municipio,% de municípios com escolas da rede estadual em tempo integral,Ano
0,2202091,1.0,2024
1,2211001,1.0,2024
2,2202737,1.0,2024
3,2207702,1.0,2024
4,2210623,1.0,2024
...,...,...,...
103,2204204,1.0,2024
104,2205904,1.0,2024
105,2200053,1.0,2024
106,2205151,1.0,2024


159 - Indicador - Nº total de matrículas da rede estadual de educação

In [225]:
# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filtered_df_24 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2024'] == 1]
filtered_df_23 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2023'] == 1]

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filtered_df_23, filtered_df_24]
combined_df = pd.concat([contar_distintos(df) for df in dataframes])

combined_df.rename(columns={'idMatricula': 'Nº total de matrículas da rede estadual de educação'}, inplace=True)
combined_df['Ano'] = combined_df['Ano'].astype(int)
display(combined_df)

,Id Municipio,Ano,Nº total de matrículas da rede estadual de educação
0,2200053,2023,335
1,2200103,2023,505
2,2200202,2023,860
3,2200251,2023,395
4,2200277,2023,163
...,...,...,...
219,2211357,2024,265
220,2211407,2024,196
221,2211506,2024,375
222,2211605,2024,369


2202 - Indicador - % de matrículas de tempo integral no ensino regular, na rede estadual

In [226]:
# Quantidade de matricula de tempo integral por estado
tabela_integral = tabela_Tempo_Integral[["idMatricula", "Id Municipio", "Ano", "tempoIntegral"]]
tab_matr = tabela_clientes[["idMatricula", "Ano", "Id Municipio", "Matricula_2024", "Matricula_2025"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tabela_integral = tabela_integral.merge(tab_matr, on=['idMatricula', 'Id Municipio', 'Ano'], how='inner')
filt_status = tabela_integral[tabela_integral['tempoIntegral'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_24 = filt_status[filt_status['Matricula_2024'] == 1]
filt_23 = filt_status[filt_status['Matricula_2025'] == 1]

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_23, filt_24]
comb_int = pd.concat([contar_distintos(df) for df in dataframes])

comb_int.rename(columns={'idMatricula': 'Qtde de matricula de tempo integral'}, inplace=True)
display(comb_int)
"----------------------------------------------------------------------------------------------------------------"
# Mesclagem para DF final
resultado_final = combined_df.merge(comb_int, on=['Id Municipio', 'Ano'], how='inner')
resultado_final['Ano'] = resultado_final['Ano'].astype(int)
"----------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
resultado_final['% de matrículas de tempo integral no ensino regular, na rede estadual'] = resultado_final['Qtde de matricula de tempo integral'] / resultado_final['Nº total de matrículas da rede estadual de educação']
resultado_final = resultado_final[["Id Municipio", "% de matrículas de tempo integral no ensino regular, na rede estadual", "Ano"]]
display(resultado_final)

,Id Municipio,Ano,Qtde de matricula de tempo integral
0,2200053,2024.0,246
1,2200202,2024.0,186
2,2200251,2024.0,240
3,2200301,2024.0,199
4,2200400,2024.0,1175
...,...,...,...
103,2211001,2024.0,6468
104,2211100,2024.0,235
105,2211407,2024.0,196
106,2211605,2024.0,369


,Id Municipio,"% de matrículas de tempo integral no ensino regular, na rede estadual",Ano
0,2200053,1.000000,2024
1,2200202,0.192149,2024
2,2200251,1.000000,2024
3,2200301,0.327303,2024
4,2200400,0.376000,2024
...,...,...,...
103,2211001,0.144556,2024
104,2211100,0.072576,2024
105,2211407,1.000000,2024
106,2211605,1.000000,2024


63 - Indicador - % de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)

In [227]:
# Quantidade de escolas de tempo integral por estado
tabela_integral_EPT = tabela_Tempo_Integral[["Id Escola", "Id Municipio", "Ano", "STATUS_INTEGRAL"]]

filt_status_TI = tabela_integral_EPT[tabela_integral_EPT['STATUS_INTEGRAL'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_25_TI = filt_status_TI[filt_status_TI['Ano'] == 2025]
filt_24_TI = filt_status_TI[filt_status_TI['Ano'] == 2024]
filt_23_TI = filt_status_TI[filt_status_TI['Ano'] == 2023]
filt_22_TI = filt_status_TI[filt_status_TI['Ano'] == 2022]

base_25_TI = filt_25_TI[["Id Escola", "Id Municipio"]]
base_24_TI = filt_24_TI[["Id Escola", "Id Municipio"]]
base_23_TI = filt_23_TI[["Id Escola", "Id Municipio"]]
base_22_TI = filt_22_TI[["Id Escola", "Id Municipio"]]

# Remover duplicatas do DataFrame, mantendo apenas a primeira ocorrência
base_25_TI = base_25_TI.drop_duplicates(keep='first')
base_24_TI = base_24_TI.drop_duplicates(keep='first')
base_23_TI = base_23_TI.drop_duplicates(keep='first')
base_22_TI = base_22_TI.drop_duplicates(keep='first')

base_23_TI = pd.concat([base_23_TI, base_22_TI])
base_23_TI['Ano'] = 2023
base_24_TI = pd.concat([base_24_TI, base_23_TI])
base_24_TI['Ano'] = 2024

# Concatenar os DataFrames
df_combinado_g = pd.concat([base_24_TI, base_23_TI], ignore_index=True)
"----------------------------------------------------------------------------------------------------------------"

# Somar os valores na coluna Status_integral por matricula (Id Matricula)
base_25_TI = base_25_TI.groupby('Id Municipio')['Id Escola'].nunique().reset_index()
base_25_TI['Ano'] = 2025
base_24_TI = base_24_TI.groupby('Id Municipio')['Id Escola'].nunique().reset_index()
base_24_TI['Ano'] = 2024
base_23_TI = base_23_TI.groupby('Id Municipio')['Id Escola'].nunique().reset_index()
base_23_TI['Ano'] = 2023
base_22_TI = base_22_TI.groupby('Id Municipio')['Id Escola'].nunique().reset_index()
base_22_TI['Ano'] = 2022

# Concatenar os DataFrames
df_combinado = pd.concat([base_24_TI, base_23_TI], ignore_index=True)
df_combinado.rename(columns={'Id Escola': 'Qtde de escolas de tempo integral'}, inplace=True)

# Exibir o DataFrame combinado
display(df_combinado)

,Id Municipio,Qtde de escolas de tempo integral,Ano
0,2200053,1,2024
1,2200103,1,2024
2,2200202,2,2024
3,2200251,1,2024
4,2200277,1,2024
...,...,...,...
314,2211001,45,2023
315,2211100,2,2023
316,2211209,3,2023
317,2211308,2,2023


In [228]:
# Tabela filtrada
tabela_clientes_TI_EPT = tabela_clientes[["Id Escola", "Id Municipio", "Matricula_2023", "Matricula_2024", "EtapaAgregada", "ordem"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filtered_TI_EPT_24 = tabela_clientes_TI_EPT[(tabela_clientes_TI_EPT['Matricula_2024'] == 1) & (tabela_clientes_TI_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_TI_EPT_24 = filtered_TI_EPT_24[["Id Escola", "Id Municipio"]]

filtered_TI_EPT_24 = filtered_TI_EPT_24.drop_duplicates(keep='first')

filtered_TI_EPT_23 = tabela_clientes_TI_EPT[(tabela_clientes_TI_EPT['Matricula_2023'] == 1) & (tabela_clientes_TI_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_TI_EPT_23 = filtered_TI_EPT_23[["Id Escola", "Id Municipio"]]

filtered_TI_EPT_23 = filtered_TI_EPT_23.drop_duplicates(keep='first')

# Concatenar os DataFrames
df_c_EPT = pd.concat([filtered_TI_EPT_24, filtered_TI_EPT_23], ignore_index=True)
#display(df_c_EPT)
"----------------------------------------------------------------------------------------------------------------"
# Relação inner join para DF final
TI_EPT = df_combinado_g.merge(df_c_EPT, how='inner')
TI_EPT = TI_EPT.drop_duplicates(keep='first')
#display(TI_EPT)
"----------------------------------------------------------------------------------------------------------------"
TI_EPT_24 = TI_EPT[TI_EPT['Ano'] == 2024]
#TI_EPT_24 = TI_EPT_24[["Id Estado", "Id Escola", "Ano"]]
TI_EPT_23 = TI_EPT[TI_EPT['Ano'] == 2023]

TI_EPT_24 = TI_EPT_24.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()
TI_EPT_23 = TI_EPT_23.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()

TI_TI_EPT = pd.concat([TI_EPT_24, TI_EPT_23])
TI_TI_EPT.rename(columns={'Id Escola': 'Qtde de escolas EPT em TI'}, inplace=True)

#display(TI_TI_EPT)

result_TI_EPT = TI_TI_EPT.merge(df_combinado, how='inner')
display(result_TI_EPT)
"----------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
result_TI_EPT['% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)'] = result_TI_EPT['Qtde de escolas EPT em TI'] / result_TI_EPT['Qtde de escolas de tempo integral']
result_TI_EPT = result_TI_EPT[["Id Municipio", "% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)", "Ano"]]
display(result_TI_EPT)

,Id Municipio,Ano,Qtde de escolas EPT em TI,Qtde de escolas de tempo integral
0,2200053,2024,1,1
1,2200103,2024,1,1
2,2200202,2024,2,2
3,2200251,2024,1,1
4,2200277,2024,1,1
...,...,...,...,...
314,2211001,2023,45,45
315,2211100,2023,2,2
316,2211209,2023,3,3
317,2211308,2023,2,2


,Id Municipio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),Ano
0,2200053,1.0,2024
1,2200103,1.0,2024
2,2200202,1.0,2024
3,2200251,1.0,2024
4,2200277,1.0,2024
...,...,...,...
314,2211001,1.0,2023
315,2211100,1.0,2023
316,2211209,1.0,2023
317,2211308,1.0,2023


2213 - Indicador - Nº de matrículas EPT - ok - A Validar

In [229]:
# Tabela filtrada
tabela_clientes_m_ept = tabela_clientes[["idMatricula", "Id Municipio", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "ordem"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filtered_df_EPTT_25 = tabela_clientes_m_ept[(tabela_clientes_m_ept['Matricula_2025'] == 1) & (tabela_clientes_m_ept['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPTT_25 = filtered_df_EPTT_25[["idMatricula", "Id Municipio", "Ano"]]
filtered_df_EPTT_24 = tabela_clientes_m_ept[(tabela_clientes_m_ept['Matricula_2024'] == 1) & (tabela_clientes_m_ept['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPTT_24 = filtered_df_EPTT_24[["idMatricula", "Id Municipio", "Ano"]]
filtered_df_EPTT_23 = tabela_clientes_m_ept[(tabela_clientes_m_ept['Matricula_2023'] == 1) & (tabela_clientes_m_ept['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPTT_23 = filtered_df_EPTT_23[["idMatricula", "Id Municipio", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "Profissional"
distinct_count_n_EPT_25 = filtered_df_EPTT_25.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()
distinct_count_n_EPT_24 = filtered_df_EPTT_24.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()
distinct_count_n_EPT_23 = filtered_df_EPTT_23.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem profissional por escola
combined_df_profissional = pd.concat([distinct_count_n_EPT_25, distinct_count_n_EPT_24, distinct_count_n_EPT_23])
combined_df_profissional.rename(columns={'idMatricula': 'Nº de matrículas EPT'}, inplace=True)
combined_df_profissional['Ano'] = combined_df_profissional['Ano'].astype(int)
display(combined_df_profissional)

,Id Municipio,Ano,Nº de matrículas EPT
0,2200053,2024,67
1,2200103,2024,236
2,2200202,2024,431
3,2200251,2024,91
4,2200277,2024,189
...,...,...,...
219,2211357,2023,131
220,2211407,2023,55
221,2211506,2023,76
222,2211605,2023,206


72 - Indicador - Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC) - ok - A Validar

In [230]:
# Tabela filtrada
tabela_clientes_prof = tabela_clientes[["idMatricula", "Id Municipio", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "EtapaAgregada", "NomeCurso", "NomeEtapa"]]

# Lista de textos a serem contados
cursos_filtrar = [
    "ADMINISTRAÇÃO",
    "CURSO TÉCNICO EM ARTESANATO COM ÊNFASE EM EMPREENDEDORISMO",
    "CONTROLE AMBIENTAL",
    "CURSO TÉCNICO EM DESENVOLVIMENTO DE SISTEMAS",
    "CURSO TÉCNICO EM ELETROTÉCNICA COM ÊNFASE EM ELETROMECÂNICA",
    "CURSO TÉCNICO EM ELETROTÉCNICA COM ÊNFASE EM HIDROGÊNIO VERDE",
    "CURSO TÉCNICO EM GUIA DE TURISMO",
    "CURSO TÉCNICO EM MARKETING DIGITAL",
    "CURSO TÉCNICO EM MINERAÇÃO",
    "PRODUÇÃO DE ÁUDIO E VÍDEO",
    "CURSO TÉCNICO EM PROGRAMAÇÃO DE JOGOS DIGITAIS",
    "CURSO TÉCNICO EM SISTEMAS DE ENERGIA RENOVÁVEL"
]

# Criar uma expressão regular que combine qualquer um dos textos
pattern = '|'.join(cursos_filtrar)

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
df_SEDUCTEC_25 = tabela_clientes_prof[(tabela_clientes_prof['Matricula_2025'] == 1) & (tabela_clientes_prof['NomeCurso'].str.contains(pattern))]
df_SEDUCTEC_25 = df_SEDUCTEC_25[["idMatricula", "Id Municipio", "Ano", "NomeCurso", "NomeEtapa"]]
df_SEDUCTEC_24 = tabela_clientes_prof[(tabela_clientes_prof['Matricula_2024'] == 1) & (tabela_clientes_prof['NomeCurso'].str.contains(pattern))]
df_SEDUCTEC_24 = df_SEDUCTEC_24[["idMatricula", "Id Municipio", "Ano", "NomeCurso", "NomeEtapa"]]
df_SEDUCTEC_23 = tabela_clientes_prof[(tabela_clientes_prof['Matricula_2023'] == 1) & (tabela_clientes_prof['NomeCurso'].str.contains(pattern))]
df_SEDUCTEC_23 = df_SEDUCTEC_23[["idMatricula", "Id Municipio", "Ano", "NomeCurso", "NomeEtapa"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "Profissional"
count_SEDUCTEC_25 = df_SEDUCTEC_25.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()
count_SEDUCTEC_24 = df_SEDUCTEC_24.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()
count_SEDUCTEC_23 = df_SEDUCTEC_23.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem profissional por escola
combined_df_SEDUCTEC = pd.concat([count_SEDUCTEC_25, count_SEDUCTEC_24, count_SEDUCTEC_23])
combined_df_SEDUCTEC['Ano'] = combined_df_SEDUCTEC['Ano'].astype(int)
combined_df_SEDUCTEC.rename(columns={'idMatricula': 'Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'}, inplace=True)
display(combined_df_SEDUCTEC)

,Id Municipio,Ano,Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)
0,2200053,2024,67
1,2200103,2024,195
2,2200202,2024,234
3,2200251,2024,68
4,2200277,2024,97
...,...,...,...
152,2211001,2023,3160
153,2211100,2023,428
154,2211209,2023,443
155,2211308,2023,137


67 - Indicador - Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais - ok - A validar

In [231]:
# Tabela filtrada
tabela_clientes_EJA = tabela_clientes[["idMatricula", "Id Municipio", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeEtapa"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "EJA"
filtered_df_EJAT_25 = tabela_clientes_EJA[(tabela_clientes_EJA['Matricula_2025'] == 1) & (tabela_clientes_EJA['NomeEtapa'].str.contains("EJA"))]
filtered_df_EJAT_25 = filtered_df_EJAT_25[["idMatricula", "Id Municipio", "Ano"]]
filtered_df_EJAT_24 = tabela_clientes_EJA[(tabela_clientes_EJA['Matricula_2024'] == 1) & (tabela_clientes_EJA['NomeEtapa'].str.contains("EJA"))]
filtered_df_EJAT_24 = filtered_df_EJAT_24[["idMatricula", "Id Municipio", "Ano"]]
filtered_df_EJAT_23 = tabela_clientes_EJA[(tabela_clientes_EJA['Matricula_2023'] == 1) & (tabela_clientes_EJA['NomeEtapa'].str.contains("EJA"))]
filtered_df_EJAT_23 = filtered_df_EJAT_23[["idMatricula", "Id Municipio", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"
EJAT_25 = filtered_df_EJAT_25.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()
EJAT_24 = filtered_df_EJAT_24.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()
EJAT_23 = filtered_df_EJAT_23.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem EJA por escola
combined_df_EJA = pd.concat([EJAT_25, EJAT_24, EJAT_23])
combined_df_EJA.rename(columns={'idMatricula': 'Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'}, inplace=True)
combined_df_EJA['Ano'] = combined_df_EJA['Ano'].astype(int)
display(combined_df_EJA)

,Id Municipio,Ano,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais
0,2200103,2024,47
1,2200202,2024,400
2,2200251,2024,23
3,2200277,2024,45
4,2200301,2024,61
...,...,...,...
204,2211357,2023,4
205,2211407,2023,22
206,2211506,2023,165
207,2211605,2023,321


2192 - Indicador - Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT) - ok - A Validar

In [232]:
# Tabela filtrada
tabela_clientes_EJA_EPT = tabela_clientes[["Id Municipio", "Id Escola", "Ano", "ativa", "ordem", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeEtapa"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "EJA"
filtered_m_EJA_EPT_25 = tabela_clientes_EJA_EPT[(tabela_clientes_EJA_EPT['Matricula_2025'] == 1) & (tabela_clientes_EJA_EPT['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_EJA_EPT['ativa'] == 1) & (tabela_clientes_EJA_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_m_EJA_EPT_25 = filtered_m_EJA_EPT_25[["Id Escola", "Id Municipio", "Ano"]]
filtered_m_EJA_EPT_24 = tabela_clientes_EJA_EPT[(tabela_clientes_EJA_EPT['Matricula_2024'] == 1) & (tabela_clientes_EJA_EPT['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_EJA_EPT['ativa'] == 1) & (tabela_clientes_EJA_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_m_EJA_EPT_24 = filtered_m_EJA_EPT_24[["Id Escola", "Id Municipio", "Ano"]]
filtered_m_EJA_EPT_23 = tabela_clientes_EJA_EPT[(tabela_clientes_EJA_EPT['Matricula_2023'] == 1) & (tabela_clientes_EJA_EPT['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_EJA_EPT['ativa'] == 1) & (tabela_clientes_EJA_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_m_EJA_EPT_23 = filtered_m_EJA_EPT_23[["Id Escola", "Id Municipio", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA" 'matriculaAtiva'
m_EJA_EPT_25 = filtered_m_EJA_EPT_25.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()
m_EJA_EPT_24 = filtered_m_EJA_EPT_24.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()
m_EJA_EPT_23 = filtered_m_EJA_EPT_23.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()

# Concatenar os DataFrames de contagem por escola
EJA_EPT = pd.concat([m_EJA_EPT_25, m_EJA_EPT_24, m_EJA_EPT_23])
EJA_EPT.rename(columns={'Id Escola': 'Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'}, inplace=True)
EJA_EPT['Ano'] = EJA_EPT['Ano'].astype(int)
display(EJA_EPT)

,Id Municipio,Ano,Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)
0,2200103,2024,1
1,2200202,2024,1
2,2200251,2024,1
3,2200277,2024,1
4,2200301,2024,2
...,...,...,...
186,2211308,2023,1
187,2211407,2023,1
188,2211506,2023,1
189,2211605,2023,1


65 - Indicador - Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)

In [233]:
# Tabela filtrada
tabela_clientes_EPT = tabela_clientes[["Id Municipio", "Id Escola", "Ano", "ativa", "ordem", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "Profissional"
filt_df_EPT_25 = tabela_clientes_EPT[(tabela_clientes_EPT['Matricula_2025'] == 1) & (tabela_clientes_EPT['ativa'] == 1) & (tabela_clientes_EPT['ordem'].isin([6, 7, 8, 13]))]
filt_df_EPT_25 = filt_df_EPT_25[["Id Municipio", "Ano", "Id Escola"]]
filt_df_EPT_24 = tabela_clientes_EPT[(tabela_clientes_EPT['Matricula_2024'] == 1) & (tabela_clientes_EPT['ativa'] == 1) & (tabela_clientes_EPT['ordem'].isin([6, 7, 8, 13]))]
filt_df_EPT_24 = filt_df_EPT_24[["Id Municipio", "Ano", "Id Escola"]]
filt_df_EPT_23 = tabela_clientes_EPT[(tabela_clientes_EPT['Matricula_2023'] == 1) & (tabela_clientes_EPT['ativa'] == 1) & (tabela_clientes_EPT['ordem'].isin([6, 7, 8, 13]))]
filt_df_EPT_23 = filt_df_EPT_23[["Id Municipio", "Ano", "Id Escola"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
filt_df_EPT_25 = filt_df_EPT_25.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()
filt_df_EPT_24 = filt_df_EPT_24.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()
filt_df_EPT_23 = filt_df_EPT_23.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()

EPT = pd.concat([filt_df_EPT_25, filt_df_EPT_24, filt_df_EPT_23])

# Renomear a coluna de contagem para diferenciar
EPT.rename(columns={'Id Escola': 'Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'}, inplace=True)
EPT['Ano'] = EPT['Ano'].astype(int)
display(EPT)

,Id Municipio,Ano,Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)
0,2200053,2024,1
1,2200103,2024,1
2,2200202,2024,3
3,2200251,2024,1
4,2200277,2024,1
...,...,...,...
219,2211357,2023,1
220,2211407,2023,1
221,2211506,2023,1
222,2211605,2023,1


1191 - Indicador - % de escolas da zona rural com oferta de EPT

In [234]:
# Tabela filtrada
tab_rural_EPT_tot = tabela_clientes[["Id Municipio", "Id Escola", "Ano", "ativa", "ordem", "localizacao", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]
tab_rural_EPT_tot = tab_rural_EPT_tot[(tab_rural_EPT_tot['ativa'] == 1) & (tab_rural_EPT_tot['localizacao'].str.contains("Rural"))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "Profissional"
filt_rural_EPTT_25 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2025'] == 1)]
filt_rural_EPTT_25 = filt_rural_EPTT_25[["Id Municipio", "Id Escola", "Ano"]]
filt_rural_EPTT_24 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2024'] == 1)]
filt_rural_EPTT_24 = filt_rural_EPTT_24[["Id Municipio", "Id Escola", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
filt_rural_EPTT_25 = filt_rural_EPTT_25.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()
filt_rural_EPTT_24 = filt_rural_EPTT_24.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()

rural_EPTT = pd.concat([filt_rural_EPTT_25, filt_rural_EPTT_24])

# Remover duplicatas com base nas colunas 'Id Estado' e 'Ano'
rural_EPTT = rural_EPTT.drop_duplicates(subset=['Id Municipio', 'Ano'])

# Renomear a coluna de contagem para diferenciar
rural_EPTT.rename(columns={'Id Escola': 'Nº de escolas da zona rural'}, inplace=True)
rural_EPTT['Ano'] = rural_EPTT['Ano'].astype(int)

In [235]:
# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "Profissional"
filt_rural_EPT_25 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2025'] == 1) & (tab_rural_EPT_tot['ordem'].isin([6, 7, 8, 13]))]
filt_rural_EPT_25 = filt_rural_EPT_25[["Id Municipio", "Id Escola", "Ano"]]
filt_rural_EPT_24 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2024'] == 1) & (tab_rural_EPT_tot['ordem'].isin([6, 7, 8, 13]))]
filt_rural_EPT_24 = filt_rural_EPT_24[["Id Municipio", "Id Escola", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
filt_rural_EPT_25 = filt_rural_EPT_25.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()
filt_rural_EPT_24 = filt_rural_EPT_24.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()

rural_EPT = pd.concat([filt_rural_EPT_24, filt_rural_EPT_25])

# Remover duplicatas com base nas colunas 'Id Estado' e 'Ano'
rural_EPT = rural_EPT.drop_duplicates(subset=['Id Municipio', 'Ano'])

# Renomear a coluna de contagem para diferenciar
rural_EPT.rename(columns={'Id Escola': 'Nº de escolas da zona rural com oferta de EPT'}, inplace=True)
rural_EPT['Ano'] = rural_EPT['Ano'].astype(int)

rural_final = rural_EPTT.merge(rural_EPT, on=['Id Municipio', 'Ano'], how='left')

rural_final['% de escolas da zona rural com oferta de EPT'] = rural_final['Nº de escolas da zona rural com oferta de EPT'] / rural_final['Nº de escolas da zona rural']
rural_final = rural_final[["Id Municipio", "% de escolas da zona rural com oferta de EPT", "Ano"]]
display(rural_final)

,Id Municipio,% de escolas da zona rural com oferta de EPT,Ano
0,2200459,1.000000,2024
1,2200905,1.000000,2024
2,2201051,NaN,2024
3,2201200,NaN,2024
4,2201606,1.000000,2024
5,2202000,1.000000,2024
6,2202075,1.000000,2024
7,2202133,1.000000,2024
8,2202307,0.500000,2024
9,2202406,1.000000,2024


2275 - Indicador - % de escolas da zona rural com oferta de Tempo Integral

In [236]:
# Quantidade de escolas da zona rural com oferta de tempo integral por estado
tabela_integral_Ru = tabela_Tempo_Integral[["Id Escola", "Id Municipio", "localizacao", "ativa", "Ano", "STATUS_INTEGRAL"]]
tabela_integral_Ru = tabela_integral_Ru[(tabela_integral_Ru['ativa'] == 1) & (tabela_integral_Ru['localizacao'].str.contains("Rural"))]

filt_status_Ru = tabela_integral_Ru[tabela_integral_Ru['STATUS_INTEGRAL'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_25_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2025]
filt_24_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2024]
filt_23_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2023]
filt_22_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2022]

base_25_Ru = filt_25_Ru[["Id Escola", "Id Municipio"]]
base_24_Ru = filt_24_Ru[["Id Escola", "Id Municipio"]]
base_23_Ru = filt_23_Ru[["Id Escola", "Id Municipio"]]
base_22_Ru = filt_22_Ru[["Id Escola", "Id Municipio"]]

# Remover duplicatas do DataFrame, mantendo apenas a primeira ocorrência
base_25_Ru = base_25_Ru.drop_duplicates(keep='first')
base_24_Ru = base_24_Ru.drop_duplicates(keep='first')
base_23_Ru = base_23_Ru.drop_duplicates(keep='first')
base_22_Ru = base_22_Ru.drop_duplicates(keep='first')

base_23_Ru = pd.concat([base_23_Ru, base_22_Ru])
base_23_Ru['Ano'] = 2023
base_24_Ru = pd.concat([base_24_Ru, base_23_Ru])
base_24_Ru['Ano'] = 2024

# Concatenar os DataFrames
df_combinado_Ru = pd.concat([base_24_Ru, base_23_Ru], ignore_index=True)
"----------------------------------------------------------------------------------------------------------------"

# Somar os valores na coluna Status_integral por matricula (Id Matricula)
base_25_Ru = base_25_Ru.groupby('Id Municipio')['Id Escola'].nunique().reset_index()
base_25_Ru['Ano'] = 2025
base_24_Ru = base_24_Ru.groupby('Id Municipio')['Id Escola'].nunique().reset_index()
base_24_Ru['Ano'] = 2024
base_23_Ru = base_23_Ru.groupby('Id Municipio')['Id Escola'].nunique().reset_index()
base_23_Ru['Ano'] = 2023

# Concatenar os DataFrames
df_combinad = pd.concat([base_24_Ru], ignore_index=True)
df_combinad.rename(columns={'Id Escola': 'Qtde de escolas da zona rural com oferta de Tempo Integral'}, inplace=True)

# Exibir o DataFrame combinado
display(df_combinad)

,Id Municipio,Qtde de escolas da zona rural com oferta de Tempo Integral,Ano
0,2200459,1,2024
1,2202000,1,2024
2,2203305,1,2024
3,2205706,1,2024
4,2206753,1,2024
5,2207603,1,2024
6,2207900,1,2024
7,2208403,2,2024
8,2211001,3,2024


In [237]:
rural_final_Ru = rural_EPTT.merge(df_combinad, on=['Id Municipio', 'Ano'], how='left')

rural_final_Ru['% de escolas da zona rural com oferta de Tempo Integral'] = rural_final_Ru['Qtde de escolas da zona rural com oferta de Tempo Integral'] / rural_final_Ru['Nº de escolas da zona rural']
rural_final_Ru = rural_final_Ru[["Id Municipio", "% de escolas da zona rural com oferta de Tempo Integral", "Ano"]]
display(rural_final_Ru)

,Id Municipio,% de escolas da zona rural com oferta de Tempo Integral,Ano
0,2200459,1.000000,2024
1,2200905,NaN,2024
2,2201051,NaN,2024
3,2201200,NaN,2024
4,2201606,NaN,2024
5,2202000,0.500000,2024
6,2202075,NaN,2024
7,2202133,NaN,2024
8,2202307,NaN,2024
9,2202406,NaN,2024


2212 - Indicador - Nº de matrículas de EJA integrado ao EPT - ok - A Validar

In [238]:
# Tabela filtrada
tabela_clientes_inte = tabela_clientes[["idMatricula", "Id Municipio", "Ano", "ativa", "ordem", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeEtapa"]]
tabela_clientes_inte = tabela_clientes_inte[(tabela_clientes_inte['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_inte['ativa'] == 1) & (tabela_clientes_inte['ordem'].isin([6, 7, 8, 13]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "EJA"
filt_df_EJA_EPT_25 = tabela_clientes_inte[(tabela_clientes_inte['Matricula_2025'] == 1)]
filt_df_EJA_EPT_25 = filt_df_EJA_EPT_25[["idMatricula", "Id Municipio", "Ano"]]
filt_df_EJA_EPT_24 = tabela_clientes_inte[(tabela_clientes_inte['Matricula_2024'] == 1)]
filt_df_EJA_EPT_24 = filt_df_EJA_EPT_24[["idMatricula", "Id Municipio", "Ano"]]
filt_df_EJA_EPT_23 = tabela_clientes_inte[(tabela_clientes_inte['Matricula_2023'] == 1)]
filt_df_EJA_EPT_23 = filt_df_EJA_EPT_23[["idMatricula", "Id Municipio", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
dist_EJA_EPT_25 = filt_df_EJA_EPT_25.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()
dist_EJA_EPT_24 = filt_df_EJA_EPT_24.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()
dist_EJA_EPT_23 = filt_df_EJA_EPT_23.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem por escola
combined_df_EJA_EPT = pd.concat([dist_EJA_EPT_25, dist_EJA_EPT_24, dist_EJA_EPT_23])
combined_df_EJA_EPT.rename(columns={'idMatricula': 'Nº de matrículas de EJA integrado ao EPT'}, inplace=True)
combined_df_EJA_EPT['Ano'] = combined_df_EJA_EPT['Ano'].astype(int)
display(combined_df_EJA_EPT)

,Id Municipio,Ano,Nº de matrículas de EJA integrado ao EPT
0,2200103,2024,41
1,2200202,2024,197
2,2200251,2024,23
3,2200277,2024,45
4,2200301,2024,61
...,...,...,...
186,2211308,2023,191
187,2211407,2023,12
188,2211506,2023,5
189,2211605,2023,131


2246 - Indicador - Nº de matrículas do Ensino Médio - ok - A VALIDAR

In [239]:
# Tabela filtrada
tabela_clientes_m = tabela_clientes[["idMatricula", "Id Municipio", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Filtro 'Ensino Médio'
tabela_clientes_m = tabela_clientes_m[
    (tabela_clientes_m['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tabela_clientes_m['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tabela_clientes_m['grupo'].str.contains('Ensino', regex=False))
]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional e Ensino Médio
filt_m_25 = tabela_clientes_m[(tabela_clientes_m['Matricula_2025'] == 1)]
filt_m_25 = filt_m_25[["idMatricula", "Id Municipio", "Ano", "STATUS_INTEGRAL"]]
filt_m_24 = tabela_clientes_m[(tabela_clientes_m['Matricula_2024'] == 1)]
filt_m_24 = filt_m_24[["idMatricula", "Id Municipio", "Ano", "STATUS_INTEGRAL"]]

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_m_25, filt_m_24]
combined_df_EM = pd.concat([contar_distintos(df) for df in dataframes])

combined_df_EM.rename(columns={'idMatricula': 'Nº de matrículas do Ensino Médio'}, inplace=True)
combined_df_EM['Ano'] = combined_df_EM['Ano'].astype(int)

display(combined_df_EM)

,Id Municipio,Ano,Nº de matrículas do Ensino Médio
0,2200053,2024,212
1,2200103,2024,198
2,2200202,2024,505
3,2200251,2024,151
4,2200277,2024,145
...,...,...,...
219,2211357,2024,157
220,2211407,2024,148
221,2211506,2024,105
222,2211605,2024,91


61 - Indicador - % de matrículas do ensino médio em tempo integral

In [240]:
# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional e Ensino Médio
filt_m_24 = tabela_clientes_m[(tabela_clientes_m['Matricula_2024'] == 1) & (tabela_clientes_m['STATUS_INTEGRAL'] == 1)]
filt_m_24 = filt_m_24[["idMatricula", "Id Municipio", "Ano"]]
filt_m_25 = tabela_clientes_m[(tabela_clientes_m['Matricula_2025'] == 1) & (tabela_clientes_m['STATUS_INTEGRAL'] == 1)]
filt_m_25 = filt_m_25[["idMatricula", "Id Municipio", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_m_25, filt_m_24]
combined_TI_EM = pd.concat([contar_distintos(df) for df in dataframes])

combined_TI_EM.rename(columns={'idMatricula': 'Nº de matrículas do Ensino Médio em tempo integral'}, inplace=True)
"--------------------------------------------------------------------------------------------------------------------------------------------------------------------"

result_TI_EM = combined_df_EM.merge(combined_TI_EM, on=['Id Municipio', 'Ano'], how='left')
result_TI_EM['Ano'] = result_TI_EM['Ano'].astype(int)
"--------------------------------------------------------------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
result_TI_EM['% de matrículas do ensino médio em tempo integral'] = result_TI_EM['Nº de matrículas do Ensino Médio em tempo integral'] / result_TI_EM['Nº de matrículas do Ensino Médio']
result_TI_EM = result_TI_EM[["Id Municipio", "% de matrículas do ensino médio em tempo integral", "Ano"]]

display(result_TI_EM)

,Id Municipio,% de matrículas do ensino médio em tempo integral,Ano
0,2200053,0.683962,2024
1,2200103,1.000000,2024
2,2200202,0.768317,2024
3,2200251,0.357616,2024
4,2200277,0.675862,2024
...,...,...,...
219,2211357,0.605096,2024
220,2211407,0.371622,2024
221,2211506,NaN,2024
222,2211605,0.395604,2024


2230 - Indicador - Nº de escolas com oferta de tempo integral no ensino médio

In [241]:
# N° De escolas com ensino médio
tabela_clientes_e = tabela_clientes[["Id Municipio", "Id Escola", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL", "tempoIntegralEscola"]]

# Filtro 'Ensino Médio'
tabela_clientes_e = tabela_clientes_e[
    (tabela_clientes_e['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tabela_clientes_e['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tabela_clientes_e['grupo'].str.contains('Ensino', regex=False)) &
    (tabela_clientes_e['tempoIntegralEscola'] == 1)
]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional e Ensino Médio
filtered_e_25 = tabela_clientes_e[(tabela_clientes_e['Matricula_2025'] == 1)]
filtered_e_25 = filtered_e_25[["Id Escola", "Id Municipio", "Ano"]]
filtered_e_24 = tabela_clientes_e[(tabela_clientes_e['Matricula_2024'] == 1)]
filtered_e_24 = filtered_e_24[["Id Escola", "Id Municipio", "Ano"]]
filtered_e_23 = tabela_clientes_e[(tabela_clientes_e['Matricula_2023'] == 1)]
filtered_e_23 = filtered_e_23[["Id Escola", "Id Municipio", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['Id Escola'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filtered_e_25, filtered_e_24, filtered_e_23]
combined_e_EM = pd.concat([contar_distintos(df) for df in dataframes])

combined_e_EM.rename(columns={'Id Escola': 'Nº de escolas com oferta de tempo integral no ensino médio'}, inplace=True)
combined_e_EM['Ano'] = combined_e_EM['Ano'].astype(int)
display(combined_e_EM)

,Id Municipio,Ano,Nº de escolas com oferta de tempo integral no ensino médio
0,2200053,2024,1
1,2200103,2024,1
2,2200202,2024,2
3,2200251,2024,1
4,2200277,2024,1
...,...,...,...
191,2211308,2023,2
192,2211357,2023,1
193,2211407,2023,1
194,2211605,2023,1


2208 - Indicador - Nº de cursos EPT ofertados - ok - A Validar

In [242]:
# Tabela filtrada
tabela_clientes_ept_of = tabela_clientes[["idMatricula", "Id Municipio", "Matricula_2023", "Matricula_2024", "EtapaAgregada", "NomeCurso", "ordem"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filtered_df_EPT_curso_24 = tabela_clientes_ept_of[(tabela_clientes_ept_of['Matricula_2024'] == 1) & (tabela_clientes_ept_of['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPT_curso_24 = filtered_df_EPT_curso_24[["idMatricula", "Id Municipio", "Matricula_2023", "Matricula_2024", "NomeCurso"]]
filtered_df_EPT_curso_23 = tabela_clientes_ept_of[(tabela_clientes_ept_of['Matricula_2023'] == 1) & (tabela_clientes_ept_of['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPT_curso_23 = filtered_df_EPT_curso_23[["idMatricula", "Id Municipio", "Matricula_2023", "Matricula_2024", "NomeCurso"]]

# Contar os valores distintos na coluna NomeCurso por escola (idEscola) para etapa agregada igual a "Profissional"
distinct_count_of_24 = filtered_df_EPT_curso_24.groupby('Id Municipio')['NomeCurso'].nunique().reset_index()
distinct_count_of_23 = filtered_df_EPT_curso_23.groupby('Id Municipio')['NomeCurso'].nunique().reset_index()

# Adicionar coluna do ano
distinct_count_of_24['Ano'] = 2024
distinct_count_of_23['Ano'] = 2023

# Renomear a coluna de contagem para diferenciar
distinct_count_of_24.rename(columns={'NomeCurso': 'Nº de cursos EPT ofertados'}, inplace=True)
distinct_count_of_23.rename(columns={'NomeCurso': 'Nº de cursos EPT ofertados'}, inplace=True)

# Concatenar os DataFrames de contagem profissional por escola
combined_df_cursos_profissional = pd.concat([distinct_count_of_24, distinct_count_of_23])
display(combined_df_cursos_profissional)

,Id Municipio,Nº de cursos EPT ofertados,Ano
0,2200053,2,2024
1,2200103,5,2024
2,2200202,7,2024
3,2200251,4,2024
4,2200277,7,2024
...,...,...,...
219,2211357,3,2023
220,2211407,3,2023
221,2211506,4,2023
222,2211605,3,2023


2232 - Indicador - Nº de matrículas da educação especial (AEE) - ok - A Validar

In [243]:
# Tabela filtrada
tabela_clientes_AEE = tabela_clientes[["idMatricula", "Id Municipio", "Ano", "NomeCurso", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]
tabela_clientes_AEE = tabela_clientes_AEE[(tabela_clientes_AEE['NomeCurso'] == "AEE")]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filt_df_AEE_25 = tabela_clientes_AEE[(tabela_clientes_AEE['Matricula_2025'] == 1)]
filt_df_AEE_25 = filt_df_AEE_25[["idMatricula", "Id Municipio", "Ano"]]
filt_df_AEE_24 = tabela_clientes_AEE[(tabela_clientes_AEE['Matricula_2024'] == 1)]
filt_df_AEE_24 = filt_df_AEE_24[["idMatricula", "Id Municipio", "Ano"]]
filt_df_AEE_23 = tabela_clientes_AEE[(tabela_clientes_AEE['Matricula_2023'] == 1)]
filt_df_AEE_23 = filt_df_AEE_23[["idMatricula", "Id Municipio", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_df_AEE_25, filt_df_AEE_24, filt_df_AEE_23]
combined_df_AEE = pd.concat([contar_distintos(df) for df in dataframes])

combined_df_AEE.rename(columns={'idMatricula': 'Nº de matrículas da educação especial (AEE)'}, inplace=True)
combined_df_AEE['Ano'] = combined_df_AEE['Ano'].astype(int)
display(combined_df_AEE)

,Id Municipio,Ano,Nº de matrículas da educação especial (AEE)
0,2200053,2024,14
1,2200202,2024,13
2,2200301,2024,19
3,2200400,2024,16
4,2200905,2024,10
...,...,...,...
64,2210706,2023,16
65,2211001,2023,1058
66,2211100,2023,21
67,2211209,2023,7


348 - Indicadores - Nº de matrículas transferidas da rede estadual para a rede municipal - ok - A Validar

In [244]:
# Tabela filtrada
tabela_clientes_E_M = tabela_clientes[["idMatricula", "Id Municipio", "Ano", "NomeEtapa", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]

Etapa_filtrar = [
    "Ensino fundamental de 9 anos - 5º Ano",
    "Ensino fundamental de 9 anos - 6º Ano",
    "Ensino fundamental de 9 anos - 7º Ano",
    "Ensino fundamental de 9 anos - 8º Ano",
    "Ensino fundamental de 9 anos - 9º Ano"
]

# Criar uma expressão regular que combine qualquer um dos textos
pattern = '|'.join(Etapa_filtrar)

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filt_df_E_M_25 = tabela_clientes_E_M[(tabela_clientes_E_M['Matricula_2025'] == 1) & (tabela_clientes_E_M['NomeEtapa'].str.contains(pattern))]
filt_df_E_M_25 = filt_df_E_M_25[["idMatricula", "Id Municipio", "Ano"]]
filt_df_E_M_24 = tabela_clientes_E_M[(tabela_clientes_E_M['Matricula_2024'] == 1) & (tabela_clientes_E_M['NomeEtapa'].str.contains(pattern))]
filt_df_E_M_24 = filt_df_E_M_24[["idMatricula", "Id Municipio", "Ano"]]
filt_df_E_M_23 = tabela_clientes_E_M[(tabela_clientes_E_M['Matricula_2023'] == 1) & (tabela_clientes_E_M['NomeEtapa'].str.contains(pattern))]
filt_df_E_M_23 = filt_df_E_M_23[["idMatricula", "Id Municipio", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Id Municipio', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_df_E_M_25, filt_df_E_M_24, filt_df_E_M_23]
combined_df_E_M = pd.concat([contar_distintos(df) for df in dataframes])

# Exibir o DataFrame combinado
combined_df_E_M = combined_df_E_M.groupby(['Id Municipio', 'Ano'])['idMatricula'].sum().reset_index()
combined_df_E_M['Ano'] = combined_df_E_M['Ano'].astype(int)

# Ordena o DataFrame pelo ano (caso não esteja ordenado)
combined_df_E_M = combined_df_E_M.sort_values(by='Ano')

# Calcula a diferença de matrículas ano a ano
combined_df_E_M['Diferença de matrículas'] = combined_df_E_M['idMatricula'].diff()
combined_df_E_M.rename(columns={'Diferença de matrículas': 'Nº de matrículas transferidas da rede estadual para a rede municipal'}, inplace=True)
combined_df_E_M = combined_df_E_M[['Id Municipio', 'Ano', 'Nº de matrículas transferidas da rede estadual para a rede municipal']]
display(combined_df_E_M)

,Id Municipio,Ano,Nº de matrículas transferidas da rede estadual para a rede municipal
0,2200103,2023,NaN
103,2208106,2023,102.0
101,2208007,2023,1002.0
99,2207900,2023,-855.0
97,2207702,2023,2252.0
...,...,...,...
88,2206100,2024,28.0
37,2202505,2024,-17.0
86,2206001,2024,-43.0
114,2208700,2024,84.0


Mesclagem de DateFrame Parcial

In [245]:
# Mesclar os DataFrames para adicionar as colunas com o 
# 159 - 2213
final_combined_df = combined_df.merge(combined_df_profissional, on=['Id Municipio', 'Ano'], how='left')
#67
final_combined_df = final_combined_df.merge(combined_df_EJA, on=['Id Municipio', 'Ano'], how='left')
#2246
final_combined_df = final_combined_df.merge(combined_df_EM, on=['Id Municipio', 'Ano'], how='left')
#2208
final_combined_df = final_combined_df.merge(combined_df_cursos_profissional, on=['Id Municipio', 'Ano'], how='left')
#2232
final_combined_df = final_combined_df.merge(combined_df_AEE, on=['Id Municipio', 'Ano'], how='left')
#72
final_combined_df = final_combined_df.merge(combined_df_SEDUCTEC, on=['Id Municipio', 'Ano'], how='left')
#2192
final_combined_df = final_combined_df.merge(EJA_EPT, on=['Id Municipio', 'Ano'], how='left')
#348
final_combined_df = final_combined_df.merge(combined_df_E_M, on=['Id Municipio', 'Ano'], how='left')
#65
final_combined_df = final_combined_df.merge(EPT, on=['Id Municipio', 'Ano'], how='left')
#2196
final_combined_df = final_combined_df.merge(matricula_aut_indigena, on=['Id Municipio', 'Ano'], how='left')
#2215
final_combined_df = final_combined_df.merge(matricula_aut_pretos, on=['Id Municipio', 'Ano'], how='left')
#2238
final_combined_df = final_combined_df.merge(base_nota_media_M, on=['Id Municipio', 'Ano'], how='left')
#402
final_combined_df = final_combined_df.merge(base_nota_media_LP, on=['Id Municipio', 'Ano'], how='left')
#1194
final_combined_df = final_combined_df.merge(nota_LP, on=['Id Municipio', 'Ano'], how='left')
#1193
final_combined_df = final_combined_df.merge(mat_M, on=['Id Municipio', 'Ano'], how='left')
#163
final_combined_df = final_combined_df.merge(EM_geral, on=['Id Municipio', 'Ano'], how='left')
#2237
final_combined_df = final_combined_df.merge(nota_media_LP, on=['Id Municipio', 'Ano'], how='left')
#381
final_combined_df = final_combined_df.merge(base_nota_media_ITA_IME, on=['Id Municipio', 'Ano'], how='left')
#382
final_combined_df = final_combined_df.merge(base_nota_media_esp_ITA_IME, on=['Id Municipio', 'Ano'], how='left')
#385
final_combined_df = final_combined_df.merge(base_nota_freq, on=['Id Municipio', 'Ano'], how='left')
#2202
final_combined_df = final_combined_df.merge(resultado_final, on=['Id Municipio', 'Ano'], how='left')
#2212
final_combined_df = final_combined_df.merge(combined_df_EJA_EPT, on=['Id Municipio', 'Ano'], how='left')
#2230
final_combined_df = final_combined_df.merge(combined_e_EM, on=['Id Municipio', 'Ano'], how='left')
#63
final_combined_df = final_combined_df.merge(result_TI_EPT, on=['Id Municipio', 'Ano'], how='left')
#153
final_combined_df = final_combined_df.merge(resultado_mun, on=['Id Municipio', 'Ano'], how='left')
#61
final_combined_df = final_combined_df.merge(result_TI_EM, on=['Id Municipio', 'Ano'], how='left')
#85
final_combined_df = final_combined_df.merge(freq_aprovativa, on=['Id Municipio', 'Ano'], how='left')
#1191
final_combined_df = final_combined_df.merge(rural_final, on=['Id Municipio', 'Ano'], how='left')
#2275
final_combined_df = final_combined_df.merge(rural_final_Ru, on=['Id Municipio', 'Ano'], how='left')
"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"
# Mudando o tipo da coluna idEscola para object
final_combined_df['Id Municipio'] = final_combined_df['Id Municipio'].astype(int).astype(str)

# Substituindo os NaN por 0 (ou outro valor apropriado)
#2230
final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'] = final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'].fillna(0)
#2212
final_combined_df['Nº de matrículas de EJA integrado ao EPT'] = final_combined_df['Nº de matrículas de EJA integrado ao EPT'].fillna(0)
#2202
final_combined_df['% de matrículas de tempo integral no ensino regular, na rede estadual'] = final_combined_df['% de matrículas de tempo integral no ensino regular, na rede estadual'].fillna(0)
#1194
final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa'] = final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa'].fillna(0)
#1193
final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Matemática'] = final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Matemática'].fillna(0)
#381
final_combined_df['Média nas avaliações nas demais disciplinas'] = final_combined_df['Média nas avaliações nas demais disciplinas'].fillna(0)
#163
final_combined_df['% de aprovação dos alunos do Ensino Médio'] = final_combined_df['% de aprovação dos alunos do Ensino Médio'].fillna(0)
#385
final_combined_df['% frequência dos estudantes inscritos nas turmas ITA/IME'] = final_combined_df['% frequência dos estudantes inscritos nas turmas ITA/IME'].fillna(0)
#382
final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'] = final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'].fillna(0)
#402
final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'] = final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'].fillna(0)
#2237
final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'].fillna(0)
#2238
final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'].fillna(0)
#2215
final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'].fillna(0)
#2196
final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'].fillna(0)
#65
final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'] = final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'].fillna(0)
#348
final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'] = final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'].fillna(0)
#2192
final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'] = final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'].fillna(0)
#72
final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'] = final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'].fillna(0)
#2232
final_combined_df['Nº de matrículas da educação especial (AEE)'] = final_combined_df['Nº de matrículas da educação especial (AEE)'].fillna(0)
#2213
final_combined_df['Nº de matrículas EPT'] = final_combined_df['Nº de matrículas EPT'].fillna(0)
#67
final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'] = final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'].fillna(0)
#2246
final_combined_df['Nº de matrículas do Ensino Médio'] = final_combined_df['Nº de matrículas do Ensino Médio'].fillna(0)
#2208
final_combined_df['Nº de cursos EPT ofertados'] = final_combined_df['Nº de cursos EPT ofertados'].fillna(0)
#153
final_combined_df['% de municípios com escolas da rede estadual em tempo integral'] = final_combined_df['% de municípios com escolas da rede estadual em tempo integral'].fillna(0)
#61
final_combined_df['% de matrículas do ensino médio em tempo integral'] = final_combined_df['% de matrículas do ensino médio em tempo integral'].fillna(0)
#1191
final_combined_df['% de escolas da zona rural com oferta de EPT'] = final_combined_df['% de escolas da zona rural com oferta de EPT'].fillna(0)
#85
final_combined_df['% de estudantes da rede estadual com frequência regular em 75%'] = final_combined_df['% de estudantes da rede estadual com frequência regular em 75%'].fillna(0)
#2195
final_combined_df['% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)'] = final_combined_df['% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)'].fillna(0)
#2275
final_combined_df['% de escolas da zona rural com oferta de Tempo Integral'] = final_combined_df['% de escolas da zona rural com oferta de Tempo Integral'].fillna(0)
"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"
#2230
final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'] = final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'].astype(int)
#2212
final_combined_df['Nº de matrículas de EJA integrado ao EPT'] = final_combined_df['Nº de matrículas de EJA integrado ao EPT'].astype(int)
#2213
final_combined_df['Nº de matrículas EPT'] = final_combined_df['Nº de matrículas EPT'].astype(int)
#67
final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'] = final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'].astype(int)
#2246
final_combined_df['Nº de matrículas do Ensino Médio'] = final_combined_df['Nº de matrículas do Ensino Médio'].astype(int)
#2208
final_combined_df['Nº de cursos EPT ofertados'] = final_combined_df['Nº de cursos EPT ofertados'].astype(int)
#2232
final_combined_df['Nº de matrículas da educação especial (AEE)'] = final_combined_df['Nº de matrículas da educação especial (AEE)'].astype(float).astype(int)
#72
final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'] = final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'].astype(float).astype(int)
#2192
final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'] = final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'].astype(float).astype(int)
#348
final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'] = final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'].astype(float).astype(int)
#65
final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'] = final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'].astype(float).astype(int)
#2196
final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'].astype(float).astype(int)
#2215
final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'].astype(float).astype(int)
#2238
final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'].astype(float).astype(int)
#2237
final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'].astype(float).astype(int)
#402
final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'] = final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'].astype(float).astype(int)
#382
final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'] = final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'].astype(float).astype(int)
#381
final_combined_df['Média nas avaliações nas demais disciplinas'] = final_combined_df['Média nas avaliações nas demais disciplinas'].astype(float).astype(int)

final_combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 32 columns):
 #   Column                                                                                      Non-Null Count  Dtype  
---  ------                                                                                      --------------  -----  
 0   Id Municipio                                                                                448 non-null    object 
 1   Ano                                                                                         448 non-null    int32  
 2   Nº total de matrículas da rede estadual de educação                                         448 non-null    int64  
 3   Nº de matrículas EPT                                                                        448 non-null    int32  
 4   Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais                 448 non-null    int32  
 5   Nº de matrículas do Ensino Médio           

66 - Indicador - % de matrículas de EPT sobre o total de matrículas - ok - A Validar

Mesclagem de DateFrame Parcial

In [246]:
# Realizar o merge entre os dois DataFrames com base nas colunas 'Id Escola' e 'Ano'
final_percent_df = pd.merge(combined_df_profissional, combined_df, on=['Id Municipio', 'Ano'])

# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
final_percent_df['% de matrículas de EPT sobre o total de matrículas'] = final_percent_df['Nº de matrículas EPT'] / final_percent_df['Nº total de matrículas da rede estadual de educação']

final_percent_df['% de matrículas de EPT sobre o total de matrículas'] = final_percent_df['% de matrículas de EPT sobre o total de matrículas'].apply(lambda x: f"{x:.4f}").astype(float)

final_percent_df['Id Municipio'] = final_percent_df['Id Municipio'].astype(str)

final_percent_df = final_percent_df[["Id Municipio", "Ano", "% de matrículas de EPT sobre o total de matrículas"]]

# Mesclagem para DF final
final_percent_df = final_combined_df.merge(final_percent_df, on=['Id Municipio', 'Ano'], how='left')
# Exibir o DataFrame final combinado
print("Resultado final combinado:")
display(final_percent_df)

Resultado final combinado:


,Id Municipio,Ano,Nº total de matrículas da rede estadual de educação,Nº de matrículas EPT,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais,Nº de matrículas do Ensino Médio,Nº de cursos EPT ofertados,Nº de matrículas da educação especial (AEE),Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC),Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT),...,"% de matrículas de tempo integral no ensino regular, na rede estadual",Nº de matrículas de EJA integrado ao EPT,Nº de escolas com oferta de tempo integral no ensino médio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),% de municípios com escolas da rede estadual em tempo integral,% de matrículas do ensino médio em tempo integral,% de estudantes da rede estadual com frequência regular em 75%,% de escolas da zona rural com oferta de EPT,% de escolas da zona rural com oferta de Tempo Integral,% de matrículas de EPT sobre o total de matrículas
0,2200053,2023,335,116,0,0,3,15,37,0,...,0.0,0,1,0.0,0.0,0.000000,0.000000,0.0,0.0,0.3463
1,2200103,2023,505,176,90,0,4,0,128,1,...,0.0,19,1,1.0,0.0,0.000000,0.000000,0.0,0.0,0.3485
2,2200202,2023,860,160,233,0,6,14,52,1,...,0.0,81,2,1.0,0.0,0.000000,0.000000,0.0,0.0,0.1860
3,2200251,2023,395,132,23,0,5,0,34,1,...,0.0,11,1,0.0,0.0,0.000000,0.000000,0.0,0.0,0.3342
4,2200277,2023,163,108,28,0,4,0,47,1,...,0.0,12,1,1.0,0.0,0.000000,0.000000,0.0,0.0,0.6626
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,2211357,2024,265,122,6,157,4,2,26,1,...,0.0,6,1,1.0,0.0,0.605096,0.960699,0.0,0.0,0.4604
444,2211407,2024,196,103,29,148,4,0,74,1,...,1.0,29,1,1.0,1.0,0.371622,0.948980,0.0,0.0,0.5255
445,2211506,2024,375,133,216,105,5,0,69,1,...,0.0,79,0,0.0,0.0,0.000000,0.954667,0.0,0.0,0.3547
446,2211605,2024,369,103,250,91,3,0,36,1,...,1.0,39,1,1.0,1.0,0.395604,0.994580,0.0,0.0,0.2791


2266 - Indicador - % de matrículas de EJA integrado ao EPT (rede estadual)

In [247]:
# Criar uma nova coluna com a divisão das colunas 'Nº de matrículas de EJA integrado ao EPT' e 'Nº total de matrículas da rede estadual de educação'
final_percent_df['% de matrículas de EJA integrado ao EPT (rede estadual)'] = final_percent_df['Nº de matrículas de EJA integrado ao EPT'] / final_percent_df['Nº total de matrículas da rede estadual de educação']

display(final_percent_df)

,Id Municipio,Ano,Nº total de matrículas da rede estadual de educação,Nº de matrículas EPT,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais,Nº de matrículas do Ensino Médio,Nº de cursos EPT ofertados,Nº de matrículas da educação especial (AEE),Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC),Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT),...,Nº de matrículas de EJA integrado ao EPT,Nº de escolas com oferta de tempo integral no ensino médio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),% de municípios com escolas da rede estadual em tempo integral,% de matrículas do ensino médio em tempo integral,% de estudantes da rede estadual com frequência regular em 75%,% de escolas da zona rural com oferta de EPT,% de escolas da zona rural com oferta de Tempo Integral,% de matrículas de EPT sobre o total de matrículas,% de matrículas de EJA integrado ao EPT (rede estadual)
0,2200053,2023,335,116,0,0,3,15,37,0,...,0,1,0.0,0.0,0.000000,0.000000,0.0,0.0,0.3463,0.000000
1,2200103,2023,505,176,90,0,4,0,128,1,...,19,1,1.0,0.0,0.000000,0.000000,0.0,0.0,0.3485,0.037624
2,2200202,2023,860,160,233,0,6,14,52,1,...,81,2,1.0,0.0,0.000000,0.000000,0.0,0.0,0.1860,0.094186
3,2200251,2023,395,132,23,0,5,0,34,1,...,11,1,0.0,0.0,0.000000,0.000000,0.0,0.0,0.3342,0.027848
4,2200277,2023,163,108,28,0,4,0,47,1,...,12,1,1.0,0.0,0.000000,0.000000,0.0,0.0,0.6626,0.073620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,2211357,2024,265,122,6,157,4,2,26,1,...,6,1,1.0,0.0,0.605096,0.960699,0.0,0.0,0.4604,0.022642
444,2211407,2024,196,103,29,148,4,0,74,1,...,29,1,1.0,1.0,0.371622,0.948980,0.0,0.0,0.5255,0.147959
445,2211506,2024,375,133,216,105,5,0,69,1,...,79,0,0.0,0.0,0.000000,0.954667,0.0,0.0,0.3547,0.210667
446,2211605,2024,369,103,250,91,3,0,36,1,...,39,1,1.0,1.0,0.395604,0.994580,0.0,0.0,0.2791,0.105691


2207 - Indicadores - % de matrículas EPT no EM sobre o total de matrículas - ok - A Validar 

In [248]:
# Criar uma nova coluna com a divisão das colunas 'Nº de matrículas de EJA integrado ao EPT' e 'Nº total de matrículas da rede estadual de educação'
final_percent_df['% de matrículas EPT no EM sobre o total de matrículas'] = final_percent_df['Nº de matrículas do Ensino Médio'] / final_percent_df['Nº total de matrículas da rede estadual de educação']

final_percent_df['% de matrículas EPT no EM sobre o total de matrículas'] = final_percent_df['% de matrículas de EPT sobre o total de matrículas'].apply(lambda x: f"{x:.4f}").astype(float)

display(final_percent_df)

,Id Municipio,Ano,Nº total de matrículas da rede estadual de educação,Nº de matrículas EPT,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais,Nº de matrículas do Ensino Médio,Nº de cursos EPT ofertados,Nº de matrículas da educação especial (AEE),Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC),Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT),...,Nº de escolas com oferta de tempo integral no ensino médio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),% de municípios com escolas da rede estadual em tempo integral,% de matrículas do ensino médio em tempo integral,% de estudantes da rede estadual com frequência regular em 75%,% de escolas da zona rural com oferta de EPT,% de escolas da zona rural com oferta de Tempo Integral,% de matrículas de EPT sobre o total de matrículas,% de matrículas de EJA integrado ao EPT (rede estadual),% de matrículas EPT no EM sobre o total de matrículas
0,2200053,2023,335,116,0,0,3,15,37,0,...,1,0.0,0.0,0.000000,0.000000,0.0,0.0,0.3463,0.000000,0.3463
1,2200103,2023,505,176,90,0,4,0,128,1,...,1,1.0,0.0,0.000000,0.000000,0.0,0.0,0.3485,0.037624,0.3485
2,2200202,2023,860,160,233,0,6,14,52,1,...,2,1.0,0.0,0.000000,0.000000,0.0,0.0,0.1860,0.094186,0.1860
3,2200251,2023,395,132,23,0,5,0,34,1,...,1,0.0,0.0,0.000000,0.000000,0.0,0.0,0.3342,0.027848,0.3342
4,2200277,2023,163,108,28,0,4,0,47,1,...,1,1.0,0.0,0.000000,0.000000,0.0,0.0,0.6626,0.073620,0.6626
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,2211357,2024,265,122,6,157,4,2,26,1,...,1,1.0,0.0,0.605096,0.960699,0.0,0.0,0.4604,0.022642,0.4604
444,2211407,2024,196,103,29,148,4,0,74,1,...,1,1.0,1.0,0.371622,0.948980,0.0,0.0,0.5255,0.147959,0.5255
445,2211506,2024,375,133,216,105,5,0,69,1,...,0,0.0,0.0,0.000000,0.954667,0.0,0.0,0.3547,0.210667,0.3547
446,2211605,2024,369,103,250,91,3,0,36,1,...,1,1.0,1.0,0.395604,0.994580,0.0,0.0,0.2791,0.105691,0.2791


In [249]:
# Selecionar automaticamente todas as colunas para value_vars, exceto as que estão em id_vars
id_vars = ['Id Municipio', 'Ano']
value_vars = [col for col in final_percent_df.columns if col not in id_vars]

# Transformar o DataFrame para a forma PIVOT
long_df = final_percent_df.melt(id_vars=id_vars, 
                                 value_vars=value_vars,
                                 var_name='Indicador', value_name='Quantidade')

# Pivotar para obter as colunas separadas para os anos 2023 e 2024
pivot_df = long_df.pivot_table(index=['Id Municipio', 'Indicador', 'Ano'], values='Quantidade').reset_index()

# Renomear as colunas para torná-las mais legíveis
pivot_df.columns.name = None
pivot_df.rename(columns={2023: '2023', 2024: '2024'}, inplace=True)

# Seleciona as colunas "CasasDecimais" e "CodigoIndicador" do dataframe especificacao_superintendência_df
resumo_calculado = tabela_especificacao_ind_O_E[['Codigo Indicador', 'Indicador', 'CasasDecimais', 'UnidadeMedida', 'TipoIndicadorEstPro']]
resumo_calculado = resumo_calculado.drop_duplicates()
# Mesclar os DataFrames para adicionar as colunas com o cálculo
pivot_df = resumo_calculado.merge(pivot_df, on='Indicador', how='left')

pivot_df['CasasDecimais'] = pivot_df['CasasDecimais'].fillna(0)
pivot_df['CasasDecimais'] = pivot_df['CasasDecimais'].astype(int)
pivot_df['Codigo Indicador'] = pivot_df['Codigo Indicador'].fillna(0)
pivot_df['Codigo Indicador'] = pivot_df['Codigo Indicador'].astype(int)
pivot_df['Ano'] = pivot_df['Ano'].fillna(0)
pivot_df['Ano'] = pivot_df['Ano'].astype(int)
pivot_df['Id Municipio'] = pivot_df['Id Municipio'].fillna(0)
pivot_df['Id Municipio'] = pivot_df['Id Municipio'].astype(str)
pivot_df = pivot_df[['Id Municipio', 'Codigo Indicador', 'Ano', 'Quantidade', 'TipoIndicadorEstPro']]

## IDENTIFICAR MOTIVO DE DUPLICAÇÃO DE LINHAS DO DATAFRAME
pivot_df = pivot_df.drop_duplicates(keep='first')

from datetime import datetime

# Capturar a data e hora atuais
data_hora_execucao = datetime.now().strftime('%d/%m/%Y %H:%M:%S')

# Extrair o ano atual
ano_atual = datetime.now().year

# Adicionar uma nova coluna ao DataFrame com a data e hora da última execução do script
pivot_df['Data Cadastro'] = data_hora_execucao# Condicional para definir 'Mes_Referente'
# Se o ano for igual ao ano atual, extrai o mês da data atual, senão atribui o valor 12
pivot_df['Mes_Referente'] = pivot_df.apply(
    lambda row: pd.to_datetime(data_hora_execucao, format='%d/%m/%Y %H:%M:%S').month 
                if row['Ano'] == ano_atual 
                else 12, axis=1
)

pivot_df['Mes_Referente'] = pivot_df['Mes_Referente'].astype(int)

pivot_df = pivot_df[['Id Municipio', 'Codigo Indicador', 'Ano', 'Quantidade', 'Data Cadastro', 'Mes_Referente', 'TipoIndicadorEstPro']]

# Exibindo a tabela final pivotada
#print("Tabela Final Pivotada:")
#display(pivot_df)
#pivot_df.info()

Preparação e definição de Base

Especificação Indicadores por Superintendência

In [16]:
# Preencher valores NaN na coluna 'Hierarquia' com string vazia
tabela_base_gerada['Hierarquia'] = tabela_base_gerada['Hierarquia'].fillna('')

# Filtro por escola
BASE_GERADA = tabela_base_gerada[(tabela_base_gerada['Hierarquia'].str.contains('Municipio'))]

# Adicionar a nova coluna 'TipoIndicadorEstPro' com a informação 'Manual' em todas as linhas

BASE_GERADA = BASE_GERADA[['Id Municipio', 'Codigo Indicador', 'Ano', 'Quantidade', 'Data Cadastro', 'Mes_Referente', 'TipoIndicadorEstPro']]

BASE_GERADA['Id Municipio'] = BASE_GERADA['Id Municipio'].fillna(0)
BASE_GERADA['Id Municipio'] = BASE_GERADA['Id Municipio'].astype(str)

# Exibir o DataFrame
display(BASE_GERADA)
BASE_GERADA.info()

,Id Municipio,Codigo Indicador,Ano,Quantidade,Data Cadastro,Mes_Referente,TipoIndicadorEstPro
6,2200053.0,2251,2023,6.400790,2024-08-27 13:00:11,12,None
7,2200103.0,2251,2023,5.623703,2024-08-27 13:00:11,12,None
8,2200202.0,2251,2023,7.285403,2024-08-27 13:00:11,12,None
9,2200251.0,2251,2023,6.048056,2024-08-27 13:00:11,12,None
10,2200277.0,2251,2023,6.136359,2024-08-27 13:00:11,12,None
...,...,...,...,...,...,...,...
27775,2211357.0,2255,2024,498.000000,2025-06-02 10:41:28,12,E
27776,2211407.0,2255,2024,506.000000,2025-06-02 10:41:28,12,E
27777,2211506.0,2255,2024,520.000000,2025-06-02 10:41:28,12,E
27778,2211605.0,2255,2024,615.000000,2025-06-02 10:41:28,12,E


<class 'pandas.core.frame.DataFrame'>
Index: 24560 entries, 6 to 27779
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Id Municipio         24560 non-null  object        
 1   Codigo Indicador     24560 non-null  object        
 2   Ano                  24560 non-null  int64         
 3   Quantidade           24560 non-null  float64       
 4   Data Cadastro        24560 non-null  datetime64[ns]
 5   Mes_Referente        24560 non-null  int64         
 6   TipoIndicadorEstPro  19740 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 1.5+ MB


In [17]:
# Mesclar os DataFrames para adicionar as colunas com o cálculo
#base_indicadores = pd.concat([BASE_GERADA, pivot_df], ignore_index=True)
base_indicadores = BASE_GERADA
base_indicadores.drop_duplicates(keep='last', inplace=True)
display(base_indicadores)

,Id Municipio,Codigo Indicador,Ano,Quantidade,Data Cadastro,Mes_Referente,TipoIndicadorEstPro
6,2200053.0,2251,2023,6.400790,2024-08-27 13:00:11,12,None
7,2200103.0,2251,2023,5.623703,2024-08-27 13:00:11,12,None
8,2200202.0,2251,2023,7.285403,2024-08-27 13:00:11,12,None
9,2200251.0,2251,2023,6.048056,2024-08-27 13:00:11,12,None
10,2200277.0,2251,2023,6.136359,2024-08-27 13:00:11,12,None
...,...,...,...,...,...,...,...
27775,2211357.0,2255,2024,498.000000,2025-06-02 10:41:28,12,E
27776,2211407.0,2255,2024,506.000000,2025-06-02 10:41:28,12,E
27777,2211506.0,2255,2024,520.000000,2025-06-02 10:41:28,12,E
27778,2211605.0,2255,2024,615.000000,2025-06-02 10:41:28,12,E


In [18]:
# O script atualiza diariamente um arquivo CSV acumulando dados e, no primeiro dia de cada mês, cria um backup mensal com os dados do último dia do mês anterior.

from datetime import datetime, timedelta
import os
import shutil

# Verificar se a pasta historico_diario existe, se não, criar
#historico_diario_path = 'historico_diario'
%run Caminhos.ipynb

# Caminhos para as pastas de histórico diário e mensal
historico_diario_path = os.path.join(pasta_hierarquia_historico, 'historico_diario')
historico_mensal_path = os.path.join(pasta_hierarquia_historico, 'historico_mensal')

# Verificar se a pasta historico_diario existe, se não, criar
if not os.path.exists(historico_diario_path):
    os.makedirs(historico_diario_path)

# Nome do arquivo CSV diário
arquivo_diario = os.path.join(historico_diario_path, 'dados_diario_municipio.csv')

# Verificar se o arquivo já existe
if os.path.exists(arquivo_diario):
    # Se o arquivo já existir, abrir em modo de apêndice e sem o cabeçalho
    base_indicadores.to_csv(arquivo_diario, mode='a', header=False, index=False)
else:
    # Se o arquivo não existir, criar um novo com o cabeçalho
    base_indicadores.to_csv(arquivo_diario, index=False)

# Verificar se hoje é o primeiro dia do mês
hoje = datetime.now()
if hoje.day == 1:
    # Obter a data do último dia do mês anterior
    ultimo_dia_mes_anterior = hoje.replace(day=1) - timedelta(days=1)
    
    # Verificar se o arquivo diário tem entradas para o último dia do mês anterior
    dados_diarios = pd.read_csv(arquivo_diario)
    dados_ultimo_dia = dados_diarios[dados_diarios['Data Cadastro'].str.startswith(ultimo_dia_mes_anterior.strftime('%Y-%m-%d'))]

    if not dados_ultimo_dia.empty:
        # Verificar se a pasta historico_mensal existe, se não, criar
        if not os.path.exists(historico_mensal_path):
            os.makedirs(historico_mensal_path)
        
        # Salvar os dados do último dia do mês anterior no arquivo mensal
        dados_ultimo_dia.to_csv(os.path.join(historico_mensal_path, f'dados_municipio{ultimo_dia_mes_anterior.strftime("%Y-%m-%d")}.csv'), index=False)

Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/
